# IG15 vs Sparse16 Action Distribution Comparisons

This notebook focuses on comparing action distributions for the two recent experiment batches:

- `configs/intervention_gate_15`
- `configs/phase4_sparse_control_16`

The notebook is config-driven: it reads the TOML files from both folders, checks which expected runs are present in the local W&B history cache, and then builds comparison plots that match the experiment design:

- IG15 entropy and topology-reward sweep
- Sparse16 flat and gated sparse-control penalty sweep
- Sparse16 flat vs gated at the same penalty
- cross-phase comparisons between IG15 topology reward and Sparse16 gated sparse-control penalty

The scalar histories can compare action-0/do-nothing usage, non-idle usage, policy entropy, and, when present, intervention-gate, joint non-idle, and illegal-action metrics.

In [17]:
from pathlib import Path
import json
import os
import re
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError("Install plotly first, for example: pip install plotly") from exc

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 180)

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file():
    TASK_DIR = cwd
elif (cwd / "Topology_Task" / "main.py").is_file():
    TASK_DIR = cwd / "Topology_Task"
elif (cwd.parent / "main.py").is_file():
    TASK_DIR = cwd.parent
else:
    raise RuntimeError("Could not locate Topology_Task/main.py from the current working directory.")

CONFIG_ROOT = TASK_DIR / "configs"
EXPERIMENT_FOLDERS = {
    "intervention_gate_15": CONFIG_ROOT / "intervention_gate_15",
    "phase4_sparse_control_16": CONFIG_ROOT / "phase4_sparse_control_16",
}
CACHE_DIR = TASK_DIR / "outputs" / "wandb_cache"
CACHE_INDEX_PATH = CACHE_DIR / "full_history_cache_index.csv"
FIG_DIR = TASK_DIR / "outputs" / "action_distribution_figures"
TRACE_CACHE_DIR = CACHE_DIR / "action_trace_tables"
for directory in [FIG_DIR, TRACE_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Task dir: {TASK_DIR}")
print(f"Cache index: {CACHE_INDEX_PATH}")
print(f"Figure dir: {FIG_DIR}")


Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache index: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history_cache_index.csv
Figure dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures


## Configuration

In [18]:
FINAL_WINDOW_STEPS = 10
SMOOTH_WINDOW = 5
PLOT_STEP_MAX_M = None
SAVE_FIGURES = True
SHOW_FIGURES = True

# Leave as None to use the first available comparison. After running the comparison-spec cell,
# set this to one of COMPARISON_SPECS["comparison_title"].
SELECTED_COMPARISON = None

# Optional exact action-id traces. Only useful for runs launched with trace_rollout_actions=true.
FETCH_TRACE_TABLES_FROM_WANDB = False
ENTITY = os.getenv("WANDB_ENTITY", "corentin-plumet-epfl")
PROJECT = os.getenv("WANDB_PROJECT", "Grid2Op")
WANDB_API_TIMEOUT = 300
TRACE_TOP_K_ACTION_IDS = 10
TRACE_TABLE_KEYS = (
    "train/rollout_action_trace",
    "train/rollout_action_trace_table",
    "test/rollout_action_trace",
    "test/rollout_action_trace_table",
    "eval/rollout_action_trace",
    "eval/rollout_action_trace_table",
)

IG15_LABELS = {
    "ig_00_phase2_base": "IG15 p000 constant entropy",
    "ig_01_entropy_decay": "IG15 p000 entropy decay",
    "ig_02_topo001_entropy_decay": "IG15 topo 0.001 + entropy decay",
    "ig_03_topo005_entropy_decay": "IG15 topo 0.005 + entropy decay",
    "ig_04_topo010_entropy_decay": "IG15 topo 0.010 + entropy decay",
}
SPARSE16_PENALTY_LABELS = {
    0.000: "p0.000",
    0.001: "p0.001",
    0.003: "p0.003",
    0.010: "p0.010",
}


## Read Experiment Configs And Cache Coverage

In [19]:
def safe_name(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    return text or "plot"


def save_figure(fig, name):
    if not SAVE_FIGURES or fig is None:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print(f"Saved: {path}")
    return path


SEED_POINT_MARKER = {
    "size": 8,
    "symbol": "circle",
    "opacity": 0.9,
    "line": {"color": "white", "width": 0.9},
}


def _bar_trace_color_map(fig):
    colors = {}
    if fig is None:
        return colors
    for trace in fig.data:
        if getattr(trace, "type", None) == "bar" and getattr(trace, "name", None) is not None:
            colors[str(trace.name)] = trace.marker.color
    return colors


def _customdata_from_columns(data, columns):
    if not columns:
        return None
    arrays = []
    for column in columns:
        values = data[column]
        if values.dtype == "object":
            values = values.astype(str)
        arrays.append(values.to_numpy())
    return np.stack(arrays, axis=-1)


def _ordered_unique(values):
    out = []
    seen = set()
    for value in values:
        key = str(value)
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out


def _seed_jitter_positions(n_points, width):
    if n_points <= 1:
        return np.zeros(n_points)
    spread = min(width * 0.28, 0.08)
    return np.linspace(-spread, spread, n_points)


def add_seed_point_overlay(fig, seed_data, *, x_col, y_col, group_col="condition_label", customdata_cols=None, hovertemplate=None):
    if fig is None or seed_data is None or seed_data.empty:
        return fig

    bar_traces = [trace for trace in fig.data if getattr(trace, "type", None) == "bar"]
    if not bar_traces:
        return fig

    grouped_axis = x_col != group_col
    category_order = _ordered_unique(
        value
        for trace in bar_traces
        for value in list(trace.x)
    )
    group_order = _ordered_unique(trace.name for trace in bar_traces)
    category_to_position = {category: idx for idx, category in enumerate(category_order)}

    if grouped_axis:
        cluster_width = 0.82
        group_width = cluster_width / max(len(group_order), 1)
        group_to_offset = {
            group: (idx - (len(group_order) - 1) / 2.0) * group_width
            for idx, group in enumerate(group_order)
        }
        bar_width = group_width * 0.86
    else:
        group_to_offset = {group: 0.0 for group in group_order}
        bar_width = 0.64

    colors = _bar_trace_color_map(fig)
    for trace in bar_traces:
        group_name = str(trace.name)
        original_x = [str(value) for value in list(trace.x)]
        trace.hovertext = original_x
        if trace.hovertemplate:
            trace.hovertemplate = trace.hovertemplate.replace("%{x}", "%{hovertext}")
        trace.x = [category_to_position[value] + group_to_offset.get(group_name, 0.0) for value in original_x]
        trace.width = bar_width

    seed_points = seed_data.copy()
    seed_points["__category_label"] = seed_points[x_col].astype(str)
    seed_points["__group_label"] = seed_points[group_col].astype(str)
    seed_points["__bar_x"] = seed_points["__category_label"].map(category_to_position)
    seed_points["__bar_x"] = seed_points["__bar_x"] + seed_points["__group_label"].map(group_to_offset).fillna(0.0)
    seed_points = seed_points.dropna(subset=["__bar_x", y_col])
    if seed_points.empty:
        fig.update_xaxes(tickmode="array", tickvals=list(range(len(category_order))), ticktext=category_order)
        return fig

    adjusted_hovertemplate = hovertemplate.replace("%{x}", "%{text}") if hovertemplate else None
    group_keys = ["__group_label", "__category_label"] if grouped_axis else ["__group_label"]
    plotted_groups = []
    for group_value, group_data in seed_points.groupby(group_keys, dropna=False, sort=False):
        if grouped_axis:
            group_name = str(group_value[0])
            category_name = str(group_value[1])
        else:
            group_name = str(group_value[0] if isinstance(group_value, tuple) else group_value)
            category_name = None
        sort_cols = [column for column in ["seed", "run_name"] if column in group_data.columns]
        group_data = group_data.sort_values(sort_cols).copy() if sort_cols else group_data.copy()
        group_data["__dot_x"] = group_data["__bar_x"].to_numpy() + _seed_jitter_positions(len(group_data), bar_width)
        marker = dict(SEED_POINT_MARKER)
        marker["color"] = colors.get(group_name, "rgba(45,45,45,0.78)")
        trace_name = f"{group_name} seeds" if category_name is None else f"{group_name} {category_name} seeds"
        plotted_groups.append(group_name)
        fig.add_trace(
            go.Scatter(
                x=group_data["__dot_x"],
                y=group_data[y_col],
                text=group_data["__category_label"],
                mode="markers",
                name=trace_name,
                legendgroup=group_name,
                showlegend=False,
                marker=marker,
                customdata=_customdata_from_columns(group_data, customdata_cols or []),
                hovertemplate=adjusted_hovertemplate,
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(category_order))),
        ticktext=category_order,
    )
    return fig

def _as_float(value, default=np.nan):
    if value is None:
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _as_bool(value):
    if isinstance(value, bool):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    lower = str(value).strip().lower()
    if lower in {"1", "true", "yes", "y", "on"}:
        return True
    if lower in {"0", "false", "no", "n", "off"}:
        return False
    return None


def _seed_from_stem(stem):
    match = re.search(r"_s(\d+)$", stem)
    return int(match.group(1)) if match else np.nan


def _family_from_stem(stem):
    return re.sub(r"_s\d+$", "", stem)


def _sparse_penalty_from_family(family):
    match = re.search(r"_p(\d{3})$", family)
    if not match:
        return np.nan
    return int(match.group(1)) / 1000.0


def _sparse_design_from_family(family):
    match = re.match(r"^sparse16_(flat|gated)_p\d{3}$", family)
    return match.group(1) if match else None


def _metadata_from_config(path, experiment):
    cfg = tomllib.loads(Path(path).read_text(encoding="utf-8"))
    args = cfg.get("args", {})
    stem = Path(path).stem
    family = _family_from_stem(stem)
    seed = _seed_from_stem(stem)
    gate_enabled = _as_bool(args.get("intervention_gate"))
    topology_reward = _as_float(args.get("topology_reward_weight"), 0.0)
    intervention_penalty = _as_float(args.get("intervention_penalty"), 0.0)
    entropy_initial = _as_float(args.get("entropy_coef"), np.nan)
    entropy_final = _as_float(args.get("entropy_coef_final"), np.nan)
    entropy_schedule = "decay" if entropy_final < entropy_initial else "constant"

    if experiment == "intervention_gate_15":
        family_label = IG15_LABELS.get(family, family.replace("_", " "))
        design = "gated"
        control_axis = "topology_reward"
        control_value = topology_reward
        control_label = f"topo {topology_reward:g}"
        comparison_group = "IG15"
    elif experiment == "phase4_sparse_control_16":
        design = _sparse_design_from_family(family) or ("gated" if gate_enabled else "flat")
        control_axis = "intervention_penalty"
        control_value = intervention_penalty if not np.isnan(intervention_penalty) else _sparse_penalty_from_family(family)
        control_label = SPARSE16_PENALTY_LABELS.get(control_value, f"p{control_value:g}")
        family_label = f"Sparse16 {design} {control_label}"
        comparison_group = "Sparse16"
    else:
        family_label = family.replace("_", " ")
        design = "gated" if gate_enabled else "flat"
        control_axis = "unknown"
        control_value = np.nan
        control_label = "unknown"
        comparison_group = experiment

    return {
        "expected_run_name": stem,
        "config_path": str(path),
        "experiment": experiment,
        "comparison_group": comparison_group,
        "family": family,
        "family_label": family_label,
        "seed": seed,
        "design": design,
        "gate_enabled": gate_enabled,
        "control_axis": control_axis,
        "control_value": control_value,
        "control_label": control_label,
        "topology_reward_weight": topology_reward,
        "intervention_penalty": intervention_penalty,
        "entropy_coef": entropy_initial,
        "entropy_coef_final": entropy_final,
        "entropy_schedule": entropy_schedule,
        "total_timesteps": args.get("total_timesteps"),
        "eval_freq": args.get("eval_freq"),
        "n_steps": args.get("n_steps"),
        "n_envs": args.get("n_envs"),
        "rollout_action_samples": int(args.get("n_steps", 0)) * int(args.get("n_envs", 0)),
    }


def read_expected_configs():
    rows = []
    for experiment, folder in EXPERIMENT_FOLDERS.items():
        if not folder.exists():
            print(f"Missing config folder: {folder}")
            continue
        for path in sorted(folder.glob("*.toml")):
            rows.append(_metadata_from_config(path, experiment))
    if not rows:
        raise RuntimeError("No expected config TOMLs found.")
    return pd.DataFrame(rows)


def load_cache_index(path=CACHE_INDEX_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {path}. Run the W&B history cache notebook first.")
    index = pd.read_csv(path)
    index = index.rename(columns={"name": "run_name", "id": "run_id"}).copy()
    for col in ["history_parquet", "history_csv"]:
        if col not in index.columns:
            index[col] = None
    index["history_parquet"] = index["history_parquet"].apply(lambda value: Path(value) if pd.notna(value) else None)
    index["history_csv"] = index["history_csv"].apply(lambda value: Path(value) if pd.notna(value) else None)
    index["has_history"] = index.apply(
        lambda row: bool(row["history_parquet"] and row["history_parquet"].exists())
        or bool(row["history_csv"] and row["history_csv"].exists()),
        axis=1,
    )
    return index[index["has_history"]].reset_index(drop=True)


EXPECTED_CONFIGS = read_expected_configs()
cache_index = load_cache_index()
selected_runs = cache_index.merge(
    EXPECTED_CONFIGS,
    left_on="run_name",
    right_on="expected_run_name",
    how="inner",
)
coverage = EXPECTED_CONFIGS.merge(
    selected_runs[["expected_run_name", "run_id", "rows", "columns"]],
    on="expected_run_name",
    how="left",
)
coverage["cached"] = coverage["run_id"].notna()

print(f"Expected configs: {len(EXPECTED_CONFIGS)}")
print(f"Cached expected runs: {coverage['cached'].sum()} / {len(coverage)}")
# display(coverage.groupby(["experiment", "family_label"], dropna=False).agg(
#     expected=("expected_run_name", "count"),
#     cached=("cached", "sum"),
#     seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique())),
# ).reset_index())

missing = coverage[~coverage["cached"]].sort_values(["experiment", "family_label", "seed"])
if not missing.empty:
    print("Missing cached histories for these expected configs:")
    display(missing[["experiment", "expected_run_name", "family_label", "seed", "control_label", "design"]])


Expected configs: 31
Cached expected runs: 30 / 31
Missing cached histories for these expected configs:


,experiment,expected_run_name,family_label,seed,control_label,design
15,phase4_sparse_control_16,sparse16_flat_p000_s0,Sparse16 flat p0.000,0,p0.000,flat


## Load Histories And Discover Action Metrics

In [20]:
def read_cached_history(row):
    parquet_path = row.get("history_parquet")
    csv_path = row.get("history_csv")
    if parquet_path and Path(parquet_path).exists():
        history = pd.read_parquet(parquet_path)
    elif csv_path and Path(csv_path).exists():
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No cached history file for {row['run_name']} ({row['run_id']})")
    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    if "_step" not in history.columns:
        if "charts/global_step" in history.columns:
            history["_step"] = history["charts/global_step"]
        elif "step" in history.columns:
            history["_step"] = history["step"]
        else:
            history["_step"] = np.arange(len(history), dtype=float)
    history["_step"] = pd.to_numeric(history["_step"], errors="coerce")
    history["step_millions"] = history["_step"] / 1_000_000
    return history


def load_cached_histories(selected):
    frames = []
    total = len(selected)
    for idx, row in enumerate(selected.to_dict("records"), start=1):
        print(f"[{idx:>3}/{total}] loading {row['run_name']}", flush=True)
        try:
            frames.append(read_cached_history(row))
        except Exception as exc:
            print(f"    skipped: {type(exc).__name__}: {exc}")
    if not frames:
        raise RuntimeError("No histories could be loaded from selected cached runs.")
    history = pd.concat(frames, ignore_index=True, sort=False).dropna(subset=["_step"])
    if PLOT_STEP_MAX_M is not None:
        history = history[history["step_millions"] <= float(PLOT_STEP_MAX_M)]
    meta_cols = [
        "run_name", "run_id", "experiment", "comparison_group", "family", "family_label", "seed",
        "design", "gate_enabled", "control_axis", "control_value", "control_label",
        "topology_reward_weight", "intervention_penalty", "entropy_schedule",
        "n_steps", "n_envs", "rollout_action_samples",
    ]
    return history.merge(selected[meta_cols], on=["run_name", "run_id"], how="left").reset_index(drop=True)


history_wide = load_cached_histories(selected_runs)
print(f"Loaded history shape: {history_wide.shape}")

ACTION_COLUMN_HINTS = [
    "frac_action_0", "entropy_agent", "intervention_gate", "nonidle_action_entropy",
    "non_idle_agents", "illegal_action",
]
metric_availability = []
for row in selected_runs.to_dict("records"):
    run_history = history_wide[history_wide["run_id"] == row["run_id"]]
    cols = list(run_history.columns)
    available = {hint: sum(hint in str(col) for col in cols) for hint in ACTION_COLUMN_HINTS}
    metric_availability.append({
        "run_name": row["run_name"],
        "family_label": row["family_label"],
        "seed": row["seed"],
        "history_rows": len(run_history),
        **available,
    })
metric_availability = pd.DataFrame(metric_availability)
print("Action metric availability by run:")
# display(metric_availability.sort_values(["family_label", "seed"]))


[  1/30] loading ig_01_entropy_decay_s0
[  2/30] loading ig_03_topo005_entropy_decay_s0
[  3/30] loading ig_04_topo010_entropy_decay_s0
[  4/30] loading ig_00_phase2_base_s0
[  5/30] loading ig_02_topo001_entropy_decay_s0
[  6/30] loading ig_03_topo005_entropy_decay_s1
[  7/30] loading ig_01_entropy_decay_s1
[  8/30] loading ig_02_topo001_entropy_decay_s1
[  9/30] loading ig_03_topo005_entropy_decay_s2
[ 10/30] loading ig_00_phase2_base_s2
[ 11/30] loading ig_01_entropy_decay_s2
[ 12/30] loading ig_00_phase2_base_s1
[ 13/30] loading ig_02_topo001_entropy_decay_s2
[ 14/30] loading ig_04_topo010_entropy_decay_s1
[ 15/30] loading ig_04_topo010_entropy_decay_s2
[ 16/30] loading sparse16_gated_p010_s0
[ 17/30] loading sparse16_flat_p010_s0
[ 18/30] loading sparse16_gated_p000_s0
[ 19/30] loading sparse16_gated_p000_s1
[ 20/30] loading sparse16_gated_p010_s1
[ 21/30] loading sparse16_flat_p010_s1
[ 22/30] loading sparse16_flat_p001_s0
[ 23/30] loading sparse16_gated_p003_s0
[ 24/30] loading 

## Convert Histories To Comparable Action Profiles

In [21]:
ID_COLS = [
    "run_name", "run_id", "experiment", "comparison_group", "family", "family_label", "seed",
    "design", "gate_enabled", "control_axis", "control_value", "control_label",
    "topology_reward_weight", "intervention_penalty", "entropy_schedule",
    "n_steps", "n_envs", "rollout_action_samples", "_step", "step_millions",
]

AGENT_FEATURE_PATTERNS = [
    {
        "feature_group": "agent_action0",
        "label_prefix": "action 0",
        "pattern": re.compile(r"^train/frac_action_0_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "agent_non_idle",
        "label_prefix": "non-idle",
        "pattern": re.compile(r"^train/frac_action_0_(agent_\d+)$"),
        "transform": lambda values: 1.0 - values,
    },
    {
        "feature_group": "agent_entropy",
        "label_prefix": "entropy",
        "pattern": re.compile(r"^train/entropy_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "agent_illegal",
        "label_prefix": "illegal",
        "pattern": re.compile(r"^train/illegal_action_rate_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "gate_intervene_frac",
        "label_prefix": "gate intervene frac",
        "pattern": re.compile(r"^train/intervention_gate_intervene_frac_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "gate_do_nothing_frac",
        "label_prefix": "gate do-nothing frac",
        "pattern": re.compile(r"^train/intervention_gate_do_nothing_frac_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "gate_prob_intervene",
        "label_prefix": "gate prob intervene",
        "pattern": re.compile(r"^train/intervention_gate_prob_intervene_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "gate_entropy",
        "label_prefix": "gate entropy",
        "pattern": re.compile(r"^train/intervention_gate_entropy_(agent_\d+)$"),
        "transform": lambda values: values,
    },
    {
        "feature_group": "nonidle_action_entropy",
        "label_prefix": "non-idle action entropy",
        "pattern": re.compile(r"^train/nonidle_action_entropy_(agent_\d+)$"),
        "transform": lambda values: values,
    },
]

SUMMARY_FEATURES = {
    "frac_any_non_idle": "train/frac_any_non_idle",
    "frac_multi_agent_non_idle": "train/frac_multi_agent_non_idle",
    "non_idle_agents_mean": "train/non_idle_agents_mean",
    "non_idle_agents_std": "train/non_idle_agents_std",
}
NON_IDLE_COUNT_PATTERN = re.compile(r"^train/non_idle_agents_count_(\d+)_frac$")

FEATURE_GROUP_ORDER = {
    "agent_non_idle": 0,
    "agent_action0": 1,
    "gate_intervene_frac": 2,
    "gate_prob_intervene": 3,
    "gate_do_nothing_frac": 4,
    "gate_entropy": 5,
    "nonidle_action_entropy": 6,
    "joint_non_idle": 7,
    "summary_non_idle": 8,
    "agent_illegal": 9,
    "agent_entropy": 10,
}


def _melt_agent_feature(history, spec):
    cols = [col for col in history.columns if spec["pattern"].match(str(col))]
    if not cols:
        return pd.DataFrame()
    data = history[ID_COLS + cols].melt(
        id_vars=ID_COLS,
        value_vars=cols,
        var_name="metric",
        value_name="raw_value",
    )
    data["raw_value"] = pd.to_numeric(data["raw_value"], errors="coerce")
    data = data.dropna(subset=["raw_value"])
    if data.empty:
        return data
    data["entity"] = data["metric"].str.extract(spec["pattern"].pattern)[0]
    data["value"] = spec["transform"](data["raw_value"])
    data["feature_group"] = spec["feature_group"]
    data["feature_key"] = data["feature_group"] + ":" + data["entity"].astype(str)
    data["feature_label"] = spec["label_prefix"] + " " + data["entity"].astype(str)
    return data.drop(columns=["raw_value"])


def _melt_joint_non_idle(history):
    cols = [col for col in history.columns if NON_IDLE_COUNT_PATTERN.match(str(col))]
    if not cols:
        return pd.DataFrame()
    data = history[ID_COLS + cols].melt(
        id_vars=ID_COLS,
        value_vars=cols,
        var_name="metric",
        value_name="value",
    )
    data["value"] = pd.to_numeric(data["value"], errors="coerce")
    data = data.dropna(subset=["value"])
    if data.empty:
        return data
    data["entity"] = data["metric"].str.extract(NON_IDLE_COUNT_PATTERN.pattern)[0].astype(int)
    data["feature_group"] = "joint_non_idle"
    data["feature_key"] = "joint_non_idle:" + data["entity"].astype(str)
    data["feature_label"] = data["entity"].astype(str) + " agents act"
    return data


def _melt_summary_features(history):
    available = {label: col for label, col in SUMMARY_FEATURES.items() if col in history.columns}
    if not available:
        return pd.DataFrame()
    data = history[ID_COLS + list(available.values())].melt(
        id_vars=ID_COLS,
        value_vars=list(available.values()),
        var_name="metric",
        value_name="value",
    )
    data["value"] = pd.to_numeric(data["value"], errors="coerce")
    data = data.dropna(subset=["value"])
    if data.empty:
        return data
    metric_to_label = {col: label for label, col in available.items()}
    data["entity"] = data["metric"].map(metric_to_label)
    data["feature_group"] = "summary_non_idle"
    data["feature_key"] = "summary_non_idle:" + data["entity"].astype(str)
    data["feature_label"] = data["entity"].astype(str).str.replace("_", " ")
    return data


def build_feature_long(history):
    frames = []
    for spec in AGENT_FEATURE_PATTERNS:
        frames.append(_melt_agent_feature(history, spec))
    frames.append(_melt_joint_non_idle(history))
    frames.append(_melt_summary_features(history))
    frames = [frame for frame in frames if frame is not None and not frame.empty]
    if not frames:
        raise RuntimeError("No action-distribution feature metrics were found in the selected histories.")
    feature_long = pd.concat(frames, ignore_index=True, sort=False)
    feature_long["feature_group_order"] = feature_long["feature_group"].map(FEATURE_GROUP_ORDER).fillna(99)
    return feature_long


def final_window_average(data, value_col, group_cols, window=FINAL_WINDOW_STEPS):
    sorted_data = data.sort_values(group_cols + ["_step"])
    tail = sorted_data.groupby(group_cols, dropna=False).tail(int(window))
    return tail.groupby(group_cols, dropna=False, as_index=False)[value_col].mean()


def rolling_mean_by_group(data, value_col, group_cols, window=SMOOTH_WINDOW):
    out = data.sort_values(group_cols + ["_step"]).copy()
    if window is None or int(window) <= 1:
        out[f"{value_col}_smooth"] = out[value_col]
    else:
        out[f"{value_col}_smooth"] = out.groupby(group_cols, dropna=False)[value_col].transform(
            lambda values: values.rolling(int(window), min_periods=1).mean()
        )
    return out


FEATURE_LONG = build_feature_long(history_wide)
# print(f"Feature rows: {len(FEATURE_LONG):,}")
# display(FEATURE_LONG.groupby(["experiment", "family_label", "feature_group"], dropna=False).size().reset_index(name="rows"))


In [22]:
PROFILE_GROUP_COLS = [
    "run_name", "run_id", "experiment", "comparison_group", "family", "family_label", "seed",
    "design", "gate_enabled", "control_axis", "control_value", "control_label",
    "topology_reward_weight", "intervention_penalty", "entropy_schedule",
    "feature_group", "feature_group_order", "feature_key", "feature_label", "entity",
]

profile_by_run = final_window_average(FEATURE_LONG, "value", PROFILE_GROUP_COLS)
ACTION_PROFILE = profile_by_run.groupby([
    "experiment", "comparison_group", "family", "family_label", "design", "gate_enabled",
    "control_axis", "control_value", "control_label", "topology_reward_weight",
    "intervention_penalty", "entropy_schedule", "feature_group", "feature_group_order",
    "feature_key", "feature_label", "entity",
], dropna=False, as_index=False).agg(
    value=("value", "mean"),
    std=("value", "std"),
    runs=("run_id", "nunique"),
    seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique())),
)

# print(f"Final-window profile rows: {len(ACTION_PROFILE):,}")
# display(ACTION_PROFILE.groupby(["experiment", "family_label", "feature_group"], dropna=False).agg(
#     features=("feature_key", "nunique"),
#     runs=("runs", "max"),
# ).reset_index())


## Built-In Comparisons

In [23]:
def profile_has_family(family):
    return bool((ACTION_PROFILE["family"] == family).any())


def add_comparison(specs, comparison_set, comparison_title, left_family, right_family, left_label, right_label, change):
    if not profile_has_family(left_family) or not profile_has_family(right_family):
        return
    specs.append({
        "comparison_set": comparison_set,
        "comparison_title": comparison_title,
        "change": change,
        "left_family": left_family,
        "right_family": right_family,
        "left_label": left_label,
        "right_label": right_label,
    })


def build_comparison_specs():
    specs = []

    add_comparison(
        specs,
        "IG15 entropy",
        "IG15: constant entropy -> entropy decay at p0.000",
        "ig_00_phase2_base",
        "ig_01_entropy_decay",
        "constant entropy",
        "entropy decay",
        "entropy_coef_final 0.02 -> 0.0",
    )
    for family, label, value in [
        ("ig_02_topo001_entropy_decay", "topo 0.001 + entropy decay", 0.001),
        ("ig_03_topo005_entropy_decay", "topo 0.005 + entropy decay", 0.005),
        ("ig_04_topo010_entropy_decay", "topo 0.010 + entropy decay", 0.010),
    ]:
        add_comparison(
            specs,
            "IG15 topology reward sweep",
            f"IG15: topology reward 0.000 -> {value:g}",
            "ig_01_entropy_decay",
            family,
            "topo 0.000 + entropy decay",
            label,
            f"topology_reward_weight 0.000 -> {value:g}",
        )

    sparse_penalties = [0.001, 0.003, 0.010]
    for design in ["flat", "gated"]:
        base = f"sparse16_{design}_p000"
        for penalty in sparse_penalties:
            family = f"sparse16_{design}_p{int(penalty * 1000):03d}"
            add_comparison(
                specs,
                f"Sparse16 {design} penalty sweep",
                f"Sparse16 {design}: p0.000 -> p{penalty:0.3f}",
                base,
                family,
                f"{design} p0.000",
                f"{design} p{penalty:0.3f}",
                f"intervention_penalty 0.000 -> {penalty:g}",
            )

    for penalty in [0.000, 0.001, 0.003, 0.010]:
        suffix = f"p{int(penalty * 1000):03d}"
        add_comparison(
            specs,
            "Sparse16 gate effect",
            f"Sparse16 {suffix}: flat -> gated",
            f"sparse16_flat_{suffix}",
            f"sparse16_gated_{suffix}",
            f"flat {suffix}",
            f"gated {suffix}",
            "intervention_gate false -> true",
        )

    cross_phase = [
        ("ig_01_entropy_decay", "sparse16_gated_p000", "p0.000", "IG entropy decay p0.000", "Sparse gated p0.000"),
        ("ig_02_topo001_entropy_decay", "sparse16_gated_p001", "p0.001", "IG topo 0.001 + entropy decay", "Sparse gated p0.001"),
        ("ig_03_topo005_entropy_decay", "sparse16_gated_p003", "p0.003-ish", "IG topo 0.005 + entropy decay", "Sparse gated p0.003"),
        ("ig_04_topo010_entropy_decay", "sparse16_gated_p010", "p0.010", "IG topo 0.010 + entropy decay", "Sparse gated p0.010"),
    ]
    for left, right, label, left_label, right_label in cross_phase:
        add_comparison(
            specs,
            "IG15 vs Sparse16 gated",
            f"IG15 vs Sparse16 gated: {label}",
            left,
            right,
            left_label,
            right_label,
            "topology_reward shaping -> sparse intervention penalty",
        )

    specs_df = pd.DataFrame(specs)
    if specs_df.empty:
        raise RuntimeError("No comparison specs could be built from the cached runs.")
    specs_df.insert(0, "comparison_id", [safe_name(title) for title in specs_df["comparison_title"]])
    return specs_df


COMPARISON_SPECS = build_comparison_specs()
# print("Available comparisons:")
# display(COMPARISON_SPECS[["comparison_set", "comparison_title", "left_label", "right_label", "change"]])


## Comparison Deltas

In [24]:
def _side_profile(spec, side):
    family = spec[f"{side}_family"]
    label = spec[f"{side}_label"]
    data = ACTION_PROFILE[ACTION_PROFILE["family"] == family].copy()
    data["side"] = side
    data["side_label"] = label
    return data


def build_comparison_deltas():
    rows = []
    for spec in COMPARISON_SPECS.to_dict("records"):
        left = _side_profile(spec, "left").set_index("feature_key")
        right = _side_profile(spec, "right").set_index("feature_key")
        for feature_key in sorted(set(left.index) & set(right.index)):
            left_row = left.loc[feature_key]
            right_row = right.loc[feature_key]
            if isinstance(left_row, pd.DataFrame):
                left_row = left_row.iloc[0]
            if isinstance(right_row, pd.DataFrame):
                right_row = right_row.iloc[0]
            left_value = float(left_row["value"])
            right_value = float(right_row["value"])
            rows.append({
                "comparison_set": spec["comparison_set"],
                "comparison_id": spec["comparison_id"],
                "comparison_title": spec["comparison_title"],
                "change": spec["change"],
                "left_label": spec["left_label"],
                "right_label": spec["right_label"],
                "feature_group": right_row["feature_group"],
                "feature_group_order": right_row["feature_group_order"],
                "feature_key": feature_key,
                "feature_label": right_row["feature_label"],
                "entity": right_row["entity"],
                "left_value": left_value,
                "right_value": right_value,
                "delta": right_value - left_value,
                "abs_delta": abs(right_value - left_value),
                "left_runs": left_row["runs"],
                "right_runs": right_row["runs"],
            })
    return pd.DataFrame(rows).sort_values(["comparison_set", "comparison_title", "feature_group_order", "feature_label"])


COMPARISON_DELTAS = build_comparison_deltas()
print(f"Delta rows: {len(COMPARISON_DELTAS):,}")

comparison_feature_coverage = COMPARISON_DELTAS.pivot_table(
    index=["comparison_set", "comparison_title"],
    columns="feature_group",
    values="feature_key",
    aggfunc="nunique",
    fill_value=0,
).reset_index()
# print("Feature groups available for each comparison, after intersecting left/right metrics:")
# display(comparison_feature_coverage)

largest_changes = COMPARISON_DELTAS.sort_values("abs_delta", ascending=False).head(60)
# print("Largest absolute changes across available action-distribution features:")
# display(largest_changes[[
#     "comparison_set", "comparison_title", "feature_group", "feature_label",
#     "left_value", "right_value", "delta", "left_runs", "right_runs",
# ]])


Delta rows: 462


## High-Level Delta Heatmaps

Positive values mean the right side of the comparison is higher than the left side.

In [25]:
def plot_delta_heatmap(feature_groups, title, save_name, comparison_sets=None, height=700):
    data = COMPARISON_DELTAS[COMPARISON_DELTAS["feature_group"].isin(feature_groups)].copy()
    if comparison_sets is not None:
        data = data[data["comparison_set"].isin(comparison_sets)]
    if data.empty:
        print(f"No data for: {title}")
        return None
    data["row_label"] = data["feature_label"]
    matrix = data.pivot_table(
        index="row_label",
        columns="comparison_title",
        values="delta",
        aggfunc="mean",
    )
    row_order = data.sort_values(["feature_group_order", "feature_label"])["row_label"].drop_duplicates().tolist()
    col_order = data["comparison_title"].drop_duplicates().tolist()
    matrix = matrix.reindex(index=row_order, columns=col_order)
    max_abs = np.nanmax(np.abs(matrix.to_numpy())) if matrix.size else 1.0
    max_abs = max(max_abs, 1e-9)
    fig = px.imshow(
        matrix,
        color_continuous_scale="RdBu",
        zmin=-max_abs,
        zmax=max_abs,
        aspect="auto",
        labels={"x": "comparison", "y": "feature", "color": "right - left"},
        title=title,
    )
    fig.update_layout(template="plotly_white", height=height, margin={"l": 150, "r": 30, "t": 80, "b": 170})
    fig.update_xaxes(tickangle=35)
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


# fig_core_delta_heatmap = plot_delta_heatmap(
#     ["agent_non_idle", "agent_action0", "summary_non_idle", "joint_non_idle"],
#     "Core action-use deltas",
#     "ig15_sparse16_core_action_delta_heatmap",
#     height=780,
# )
# fig_gate_delta_heatmap = plot_delta_heatmap(
#     ["gate_intervene_frac", "gate_prob_intervene", "gate_do_nothing_frac", "gate_entropy", "nonidle_action_entropy"],
#     "Intervention-gate deltas where metrics are available",
#     "ig15_sparse16_gate_delta_heatmap",
#     height=760,
# )
# fig_risk_entropy_delta_heatmap = plot_delta_heatmap(
#     ["agent_illegal", "agent_entropy"],
#     "Illegal-action and policy-entropy deltas",
#     "ig15_sparse16_illegal_entropy_delta_heatmap",
#     height=650,
# )


## Control-Sweep Views

These plots show final-window behavior as a function of control strength, which is often more readable than pairwise deltas.

In [26]:
def plot_control_sweep(feature_group="agent_non_idle", experiments=None, title=None, save_name=None):
    data = ACTION_PROFILE[ACTION_PROFILE["feature_group"] == feature_group].copy()
    if experiments is not None:
        data = data[data["experiment"].isin(experiments)]
    if data.empty:
        print(f"No profile data for feature group: {feature_group}")
        return None
    data["series"] = data.apply(
        lambda row: f"{row['comparison_group']} {row['design']} {row['entropy_schedule']}",
        axis=1,
    )
    fig = px.line(
        data.sort_values(["comparison_group", "design", "control_value", "entity"]),
        x="control_value",
        y="value",
        color="series",
        line_dash="entity",
        markers=True,
        facet_col="experiment",
        facet_col_wrap=1,
        hover_data=["family_label", "runs", "seeds", "std"],
        labels={"control_value": "control strength", "value": feature_group.replace("_", " "), "series": "series"},
        title=title or f"Control sweep: {feature_group.replace('_', ' ')}",
    )
    if feature_group in {"agent_non_idle", "agent_action0", "gate_intervene_frac", "gate_prob_intervene", "gate_do_nothing_frac", "joint_non_idle", "summary_non_idle", "agent_illegal"}:
        fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=620, width=1250, hovermode="x unified")
    save_figure(fig, save_name or f"control_sweep_{feature_group}")
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_sweep_non_idle = plot_control_sweep("agent_non_idle", title="Control sweep: non-idle action fraction")
fig_sweep_action0 = plot_control_sweep("agent_action0", title="Control sweep: action-0/do-nothing fraction")
fig_sweep_entropy = plot_control_sweep("agent_entropy", title="Control sweep: policy entropy")
fig_sweep_gate = plot_control_sweep("gate_intervene_frac", title="Control sweep: intervention-gate intervene fraction")


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/control_sweep_agent_non_idle.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/control_sweep_agent_action0.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/control_sweep_agent_entropy.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/control_sweep_gate_intervene_frac.html


## Selected Comparison Dashboard

In [27]:
def resolve_selected_comparison(title=SELECTED_COMPARISON):
    if title is None:
        # Prefer a direct cross-phase comparison because this notebook focuses on IG15 vs Sparse16.
        preferred = COMPARISON_SPECS[COMPARISON_SPECS["comparison_set"] == "IG15 vs Sparse16 gated"]
        title = preferred.iloc[0]["comparison_title"] if not preferred.empty else COMPARISON_SPECS.iloc[0]["comparison_title"]
    matches = COMPARISON_SPECS[COMPARISON_SPECS["comparison_title"] == title]
    if matches.empty:
        raise ValueError(f"Unknown comparison title: {title!r}")
    return matches.iloc[0].to_dict()


def comparison_profile(spec):
    return pd.concat([_side_profile(spec, "left"), _side_profile(spec, "right")], ignore_index=True, sort=False)


def _panel_bar(fig, profile, feature_group, row, col, title, colors):
    data = profile[profile["feature_group"] == feature_group].copy()
    if data.empty:
        fig.add_annotation(text=f"No {title} data", row=row, col=col, showarrow=False)
        return
    for side_label, side_data in data.groupby("side_label", sort=False):
        fig.add_trace(
            go.Bar(
                x=side_data["entity"].astype(str),
                y=side_data["value"],
                name=side_label,
                legendgroup=side_label,
                showlegend=(row == 1 and col == 1),
                marker_color=colors.get(side_label),
                customdata=np.stack([side_data["runs"], side_data["std"].fillna(0.0)], axis=-1),
                hovertemplate="%{x}<br>value=%{y:.4f}<br>runs=%{customdata[0]}<br>std=%{customdata[1]:.4f}<extra></extra>",
            ),
            row=row,
            col=col,
        )
    fig.update_yaxes(rangemode="tozero", row=row, col=col)


def plot_selected_comparison_dashboard(title=SELECTED_COMPARISON):
    spec = resolve_selected_comparison(title)
    profile = comparison_profile(spec)
    colors = {spec["left_label"]: "#1f77b4", spec["right_label"]: "#d62728"}
    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=(
            "Non-idle fraction by agent",
            "Action 0 fraction by agent",
            "Policy entropy by agent",
            "Gate intervene fraction",
            "Joint non-idle distribution",
            "Illegal action rate by agent",
        ),
    )
    panels = [
        ("agent_non_idle", 1, 1, "non-idle fraction"),
        ("agent_action0", 1, 2, "action 0 fraction"),
        ("agent_entropy", 1, 3, "entropy"),
        ("gate_intervene_frac", 2, 1, "gate intervene fraction"),
        ("joint_non_idle", 2, 2, "joint non-idle"),
        ("agent_illegal", 2, 3, "illegal action rate"),
    ]
    for feature_group, row, col, panel_title in panels:
        _panel_bar(fig, profile, feature_group, row, col, panel_title, colors)
    fig.update_layout(
        title=f"{spec['comparison_title']}<br><sup>{spec['left_label']} -> {spec['right_label']}</sup>",
        template="plotly_white",
        barmode="group",
        height=790,
        width=1450,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
        margin={"l": 70, "r": 30, "t": 115, "b": 60},
    )
    save_figure(fig, f"selected_comparison_dashboard_{spec['comparison_id']}")
    if SHOW_FIGURES:
        fig.show()
    return fig


selected_spec = resolve_selected_comparison()
print("Selected comparison:")
display(pd.DataFrame([selected_spec]))
fig_selected_dashboard = plot_selected_comparison_dashboard(selected_spec["comparison_title"])


Selected comparison:


,comparison_id,comparison_set,comparison_title,change,left_family,right_family,left_label,right_label
0,IG15_vs_Sparse16_gated_p0.000,IG15 vs Sparse16 gated,IG15 vs Sparse16 gated: p0.000,topology_reward shaping -> sparse intervention...,ig_01_entropy_decay,sparse16_gated_p000,IG entropy decay p0.000,Sparse gated p0.000


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/selected_comparison_dashboard_IG15_vs_Sparse16_gated_p0.000.html


## Selected Comparison Time Series

In [28]:
def _feature_timeseries_for_spec(spec, feature_group):
    frames = []
    for side in ["left", "right"]:
        family = spec[f"{side}_family"]
        label = spec[f"{side}_label"]
        data = FEATURE_LONG[(FEATURE_LONG["family"] == family) & (FEATURE_LONG["feature_group"] == feature_group)].copy()
        if data.empty:
            continue
        data["side"] = side
        data["side_label"] = label
        frames.append(data)
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True, sort=False)
    grouped = combined.groupby(["side_label", "entity", "_step", "step_millions"], dropna=False, as_index=False)["value"].mean()
    return rolling_mean_by_group(grouped, "value", ["side_label", "entity"])


def plot_selected_timeseries(feature_group="agent_non_idle", title=SELECTED_COMPARISON):
    spec = resolve_selected_comparison(title)
    data = _feature_timeseries_for_spec(spec, feature_group)
    if data.empty:
        print(f"No time-series data for {feature_group} in selected comparison.")
        return None
    fig = px.line(
        data,
        x="step_millions",
        y="value_smooth",
        color="side_label",
        line_dash="entity",
        labels={"step_millions": "steps (M)", "value_smooth": feature_group.replace("_", " "), "side_label": "run side"},
        title=f"{spec['comparison_title']}: {feature_group.replace('_', ' ')} over training",
    )
    if feature_group in {"agent_non_idle", "agent_action0", "gate_intervene_frac", "gate_prob_intervene", "gate_do_nothing_frac", "agent_illegal"}:
        fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=560, width=1250, hovermode="x unified")
    save_figure(fig, f"selected_timeseries_{feature_group}_{spec['comparison_id']}")
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_selected_non_idle_ts = plot_selected_timeseries("agent_non_idle", selected_spec["comparison_title"])
fig_selected_action0_ts = plot_selected_timeseries("agent_action0", selected_spec["comparison_title"])
fig_selected_gate_ts = plot_selected_timeseries("gate_intervene_frac", selected_spec["comparison_title"])
fig_selected_entropy_ts = plot_selected_timeseries("agent_entropy", selected_spec["comparison_title"])


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/selected_timeseries_agent_non_idle_IG15_vs_Sparse16_gated_p0.000.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/selected_timeseries_agent_action0_IG15_vs_Sparse16_gated_p0.000.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/selected_timeseries_gate_intervene_frac_IG15_vs_Sparse16_gated_p0.000.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/selected_timeseries_agent_entropy_IG15_vs_Sparse16_gated_p0.000.html


## Last-5-Logged Action-0 Fraction By Agent For All Runs

These two plots use the same style as the selected-comparison dashboard, but compare every cached run from each folder at once:

- one grouped bar plot for `intervention_gate_15`
- one grouped bar plot for `phase4_sparse_control_16`

For each `run × agent`, the notebook averages the last 5 cached `train/frac_action_0_agent_*` values near the end of training. This is more stable than using only the final logged point. No new rollout is run.

In [29]:
ACTION0_AVERAGE_LAST_N_LOGGED = 5


def action0_final_rows():
    data = FEATURE_LONG[FEATURE_LONG["feature_group"] == "agent_action0"].copy()
    if data.empty:
        return data

    # Average the last few cached training values for each run and agent.
    data = data.sort_values(["run_id", "entity", "_step"])
    data = data.groupby(["run_id", "entity"], dropna=False).tail(int(ACTION0_AVERAGE_LAST_N_LOGGED)).copy()

    data["rollout_action_samples"] = pd.to_numeric(data["rollout_action_samples"], errors="coerce")
    missing_samples = data["rollout_action_samples"].isna() | (data["rollout_action_samples"] <= 0)
    if missing_samples.any():
        fallback_samples = (
            pd.to_numeric(data.get("n_steps"), errors="coerce").fillna(0)
            * pd.to_numeric(data.get("n_envs"), errors="coerce").fillna(0)
        )
        data.loc[missing_samples, "rollout_action_samples"] = fallback_samples[missing_samples]

    summary_cols = [
        "run_name", "run_id", "experiment", "comparison_group", "family", "family_label",
        "seed", "design", "control_axis", "control_value", "control_label", "entity",
        "n_steps", "n_envs", "rollout_action_samples",
    ]
    data = data.groupby(summary_cols, dropna=False, as_index=False).agg(
        final_fraction_action0=("value", "mean"),
        std_last_fraction_action0=("value", "std"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("value", "count"),
    )
    data["std_last_fraction_action0"] = data["std_last_fraction_action0"].fillna(0.0)
    data["final_action0_count"] = (data["final_fraction_action0"] * data["rollout_action_samples"]).round().astype("Int64")
    data["std_last_action0_count"] = (data["std_last_fraction_action0"] * data["rollout_action_samples"]).round().astype("Int64")
    data["agent"] = data["entity"].astype(str)
    data["run_label"] = data["run_name"]
    data["folder_label"] = data["experiment"].map({
        "intervention_gate_15": "Intervention gate 15",
        "phase4_sparse_control_16": "Phase4 sparse control 16",
    }).fillna(data["experiment"])
    data["agent"] = pd.Categorical(data["agent"], categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    return data.dropna(subset=["final_fraction_action0"])


ACTION0_FINAL_LONG = action0_final_rows()
ACTION0_COUNT_LONG = ACTION0_FINAL_LONG  # Backward-compatible name for exported table code.

ACTION0_COUNT_SUMMARY = ACTION0_FINAL_LONG[[
    "experiment",
    "run_name",
    "family_label",
    "seed",
    "agent",
    "final_fraction_action0",
    "final_action0_count",
    "final_step_millions",
    "rollout_action_samples",
]].sort_values(["experiment", "family_label", "seed", "agent"]).reset_index(drop=True)

# print(f"Action-0 fraction averaged over the last {ACTION0_AVERAGE_LAST_N_LOGGED} logged points by run and agent:")
# display(ACTION0_COUNT_SUMMARY)


def plot_action0_fraction_all_runs(experiment, title, save_name):
    data = ACTION0_FINAL_LONG[ACTION0_FINAL_LONG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No final action-0 data for {experiment}")
        return None

    data = data.sort_values(["family_label", "seed", "agent"])
    fig = px.bar(
        data,
        x="agent",
        y="final_fraction_action0",
        color="run_label",
        barmode="group",
        hover_data=[
            "family_label",
            "seed",
            "final_action0_count",
            "final_step_millions",
            "rollout_action_samples",
        ],
        labels={
            "agent": "agent",
            "final_fraction_action0": f"mean action 0 fraction over last {ACTION0_AVERAGE_LAST_N_LOGGED} logged",
            "run_label": "run",
        },
        title=title,
    )
    fig.update_yaxes(range=[0, 1], title_text="action 0 fraction")
    fig.update_layout(
        template="plotly_white",
        height=620,
        width=1450,
        bargap=0.18,
        bargroupgap=0.04,
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 70, "r": 340, "t": 90, "b": 70},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_action0_fraction_ig15_all_runs = plot_action0_fraction_all_runs(
    "intervention_gate_15",
    f"Intervention gate 15: action 0 fraction by agent for all runs (mean of last {ACTION0_AVERAGE_LAST_N_LOGGED} logged)",
    "final_action0_fraction_by_agent_all_runs_intervention_gate_15",
)
fig_action0_fraction_sparse16_all_runs = plot_action0_fraction_all_runs(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: action 0 fraction by agent for all runs (mean of last {ACTION0_AVERAGE_LAST_N_LOGGED} logged)",
    "final_action0_fraction_by_agent_all_runs_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_action0_fraction_by_agent_all_runs_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_action0_fraction_by_agent_all_runs_phase4_sparse_control_16.html


## Seed-Aggregated Last-5-Logged Action-0 Fraction By Agent

These two plots aggregate runs that share the same configuration and differ only by seed. Each bar is the mean action-0 fraction over cached seeds, where each run first averages its last 5 logged `train/frac_action_0_agent_*` values.

In [30]:
def _seed_list(values):
    return sorted(pd.Series(values).dropna().astype(int).unique().tolist())


def seed_aggregated_action0_rows():
    if ACTION0_FINAL_LONG.empty:
        return pd.DataFrame()
    grouped = ACTION0_FINAL_LONG.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "agent",
        ],
        dropna=False,
        observed=True,
        as_index=False,
    ).agg(
        mean_final_fraction_action0=("final_fraction_action0", "mean"),
        std_final_fraction_action0=("final_fraction_action0", "std"),
        mean_final_action0_count=("final_action0_count", "mean"),
        std_final_action0_count=("final_action0_count", "std"),
        n_seeds=("seed", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_final_fraction_action0"] = grouped["std_final_fraction_action0"].fillna(0.0)
    grouped["std_final_action0_count"] = grouped["std_final_action0_count"].fillna(0.0)
    grouped["agent"] = pd.Categorical(grouped["agent"], categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    grouped["condition_label"] = grouped["family_label"]
    return grouped.sort_values(["experiment", "control_value", "design", "family_label", "agent"])


ACTION0_SEED_AGG = seed_aggregated_action0_rows()
# print(f"Seed-aggregated action-0 fraction by condition and agent (run mean of last {ACTION0_AVERAGE_LAST_N_LOGGED} logged):")
# display(ACTION0_SEED_AGG)


def plot_action0_fraction_seed_aggregated(experiment, title, save_name):
    data = ACTION0_SEED_AGG[ACTION0_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No seed-aggregated final action-0 data for {experiment}")
        return None

    data = data.sort_values(["control_value", "design", "condition_label", "agent"])
    fig = px.bar(
        data,
        x="agent",
        y="mean_final_fraction_action0",
        color="condition_label",
        error_y="std_final_fraction_action0",
        barmode="group",
        hover_data={
            "condition_label": True,
            "control_label": True,
            "design": True,
            "n_seeds": True,
            "seeds": True,
            "mean_final_action0_count": ":.0f",
            "std_final_action0_count": ":.0f",
            "mean_final_fraction_action0": ":.4f",
            "std_final_fraction_action0": ":.4f",
        },
        labels={
            "agent": "agent",
            "mean_final_fraction_action0": "mean final action 0 fraction",
            "condition_label": "condition",
        },
        title=title,
    )

    seed_points = ACTION0_FINAL_LONG[ACTION0_FINAL_LONG["experiment"] == experiment].copy()
    if not seed_points.empty:
        seed_points["condition_label"] = seed_points["family_label"]
        seed_points = seed_points.sort_values(["control_value", "design", "condition_label", "agent", "seed"])
        add_seed_point_overlay(
            fig,
            seed_points,
            x_col="agent",
            y_col="final_fraction_action0",
            group_col="condition_label",
            customdata_cols=[
                "run_name",
                "seed",
                "condition_label",
                "final_fraction_action0",
                "final_action0_count",
                "averaged_logged_points",
            ],
            hovertemplate=(
                "seed run=%{customdata[0]}<br>"
                "seed=%{customdata[1]}<br>"
                "condition=%{customdata[2]}<br>"
                "agent=%{x}<br>"
                "seed action-0 fraction=%{y:.4f}<br>"
                "action-0 count=%{customdata[4]}<br>"
                "logged points=%{customdata[5]}<extra></extra>"
            ),
        )

    fig.update_yaxes(range=[0, 1], title_text="mean final action 0 fraction")
    fig.update_layout(
        template="plotly_white",
        height=620,
        width=1450,
        bargap=0.18,
        bargroupgap=0.04,
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 70, "r": 340, "t": 90, "b": 70},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_action0_fraction_ig15_seed_agg = plot_action0_fraction_seed_aggregated(
    "intervention_gate_15",
    f"Intervention gate 15: seed-aggregated action 0 fraction by agent (run mean of last {ACTION0_AVERAGE_LAST_N_LOGGED} logged)",
    "seed_aggregated_final_action0_fraction_by_agent_intervention_gate_15",
)
fig_action0_fraction_sparse16_seed_agg = plot_action0_fraction_seed_aggregated(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: seed-aggregated action 0 fraction by agent (run mean of last {ACTION0_AVERAGE_LAST_N_LOGGED} logged)",
    "seed_aggregated_final_action0_fraction_by_agent_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/seed_aggregated_final_action0_fraction_by_agent_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/seed_aggregated_final_action0_fraction_by_agent_phase4_sparse_control_16.html


## Survival vs Action-0 / Non-Idle Over Time

These aligned panels compare seed-aggregated survival with seed-aggregated action behavior for each condition. The behavior panels first average action-0 / non-idle fraction across agents within each run, then average those run-level curves across cached seeds.

In [31]:
SURVIVAL_ACTION_BEHAVIOR_SMOOTH_WINDOW = 5
SURVIVAL_METRIC_CANDIDATES = [
    "test/charts/episodic_survival",
    "test/episodic_survival",
    "validation/episodic_survival",
    "charts/episodic_survival",
    "train_eval/charts/episodic_survival",
    "train_eval/episodic_survival",
]

try:
    _seed_list
except NameError:
    def _seed_list(values):
        return sorted(pd.Series(values).dropna().astype(int).unique().tolist())


def _survival_scale(values):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def seed_aggregated_survival_rows():
    frames = []
    for run_id, run_history in history_wide.groupby("run_id", sort=False):
        metric = next(
            (
                candidate for candidate in SURVIVAL_METRIC_CANDIDATES
                if candidate in run_history.columns and run_history[candidate].notna().any()
            ),
            None,
        )
        if metric is None:
            continue

        values = pd.to_numeric(run_history[metric], errors="coerce")
        frame = run_history[[
            "run_name",
            "run_id",
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "seed",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "_step",
            "step_millions",
        ]].copy()
        frame["survival_pct"] = values * _survival_scale(values)
        frame["metric_used"] = metric
        frames.append(frame.dropna(subset=["survival_pct", "_step", "step_millions"]))

    if not frames:
        return pd.DataFrame()

    survival = pd.concat(frames, ignore_index=True, sort=False)
    grouped = survival.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "_step",
            "step_millions",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_survival_pct=("survival_pct", "mean"),
        std_survival_pct=("survival_pct", "std"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        metrics_used=("metric_used", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_survival_pct"] = grouped["std_survival_pct"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "_step"])

    if SURVIVAL_ACTION_BEHAVIOR_SMOOTH_WINDOW and int(SURVIVAL_ACTION_BEHAVIOR_SMOOTH_WINDOW) > 1:
        grouped["mean_survival_pct_smooth"] = grouped.groupby(
            ["experiment", "family"],
            dropna=False,
        )["mean_survival_pct"].transform(
            lambda values: values.rolling(int(SURVIVAL_ACTION_BEHAVIOR_SMOOTH_WINDOW), min_periods=1).mean()
        )
        grouped["std_survival_pct_smooth"] = grouped.groupby(
            ["experiment", "family"],
            dropna=False,
        )["std_survival_pct"].transform(
            lambda values: values.rolling(int(SURVIVAL_ACTION_BEHAVIOR_SMOOTH_WINDOW), min_periods=1).mean()
        )
    else:
        grouped["mean_survival_pct_smooth"] = grouped["mean_survival_pct"]
        grouped["std_survival_pct_smooth"] = grouped["std_survival_pct"]
    return grouped


SURVIVAL_SEED_AGG = seed_aggregated_survival_rows()
# print("Seed-aggregated survival time series:")
# display(SURVIVAL_SEED_AGG.groupby(["experiment", "family_label"], dropna=False).agg(
#     points=("_step", "count"),
#     n_seeds=("n_seeds", "max"),
#     seeds=("seeds", "first"),
#     metrics=("metrics_used", "first"),
# ).reset_index())


ACTION_BEHAVIOR_SMOOTH_WINDOW = 5


def seed_aggregated_action_behavior_rows():
    behavior = FEATURE_LONG[FEATURE_LONG["feature_group"].isin(["agent_action0", "agent_non_idle"])].copy()
    if behavior.empty:
        return pd.DataFrame()

    # First collapse the three agents inside each run, so each run contributes one
    # action-0 and one non-idle curve to the seed aggregate.
    per_run = behavior.groupby(
        [
            "run_name",
            "run_id",
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "seed",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "feature_group",
            "_step",
            "step_millions",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        run_mean_value=("value", "mean"),
        agents=("entity", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )

    grouped = per_run.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "feature_group",
            "_step",
            "step_millions",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_value=("run_mean_value", "mean"),
        std_value=("run_mean_value", "std"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_value"] = grouped["std_value"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped["metric_label"] = grouped["feature_group"].map({
        "agent_action0": "action 0 fraction",
        "agent_non_idle": "non-idle fraction",
    })
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "feature_group", "_step"])

    if ACTION_BEHAVIOR_SMOOTH_WINDOW and int(ACTION_BEHAVIOR_SMOOTH_WINDOW) > 1:
        grouped["mean_value_smooth"] = grouped.groupby(
            ["experiment", "family", "feature_group"],
            dropna=False,
        )["mean_value"].transform(lambda values: values.rolling(int(ACTION_BEHAVIOR_SMOOTH_WINDOW), min_periods=1).mean())
        grouped["std_value_smooth"] = grouped.groupby(
            ["experiment", "family", "feature_group"],
            dropna=False,
        )["std_value"].transform(lambda values: values.rolling(int(ACTION_BEHAVIOR_SMOOTH_WINDOW), min_periods=1).mean())
    else:
        grouped["mean_value_smooth"] = grouped["mean_value"]
        grouped["std_value_smooth"] = grouped["std_value"]
    return grouped


ACTION_BEHAVIOR_SEED_AGG = seed_aggregated_action_behavior_rows()
# print("Seed-aggregated action behavior time series:")
# display(ACTION_BEHAVIOR_SEED_AGG.groupby(["experiment", "family_label", "metric_label"], dropna=False).agg(
#     points=("_step", "count"),
#     n_seeds=("n_seeds", "max"),
#     seeds=("seeds", "first"),
# ).reset_index())


def plot_survival_vs_action_behavior(experiment, title, save_name):
    survival = SURVIVAL_SEED_AGG[SURVIVAL_SEED_AGG["experiment"] == experiment].copy()
    behavior = ACTION_BEHAVIOR_SEED_AGG[ACTION_BEHAVIOR_SEED_AGG["experiment"] == experiment].copy()
    if survival.empty or behavior.empty:
        print(f"Missing survival or action-behavior data for {experiment}")
        return None

    conditions = pd.concat([
        survival[["family", "condition_label", "control_value", "design"]],
        behavior[["family", "condition_label", "control_value", "design"]],
    ], ignore_index=True).drop_duplicates()
    conditions = conditions.sort_values(["control_value", "design", "condition_label"])
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {row.family: palette[idx % len(palette)] for idx, row in enumerate(conditions.itertuples(index=False))}

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.055,
        subplot_titles=(
            "Episodic survival",
            "Mean action-0 fraction across agents",
            "Mean non-idle fraction across agents",
        ),
    )

    for _, row in conditions.iterrows():
        family = row["family"]
        label = row["condition_label"]
        color = colors[family]

        survival_data = survival[survival["family"] == family].sort_values("step_millions")
        if not survival_data.empty:
            fig.add_trace(
                go.Scatter(
                    x=survival_data["step_millions"],
                    y=survival_data["mean_survival_pct_smooth"],
                    mode="lines",
                    name=label,
                    legendgroup=family,
                    showlegend=True,
                    line={"color": color, "width": 3},
                    customdata=np.stack([
                        survival_data["n_seeds"],
                        survival_data["seeds"].astype(str),
                        survival_data["metrics_used"].astype(str),
                    ], axis=-1),
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        "step=%{x:.2f}M<br>"
                        "survival=%{y:.2f}%<br>"
                        "seeds=%{customdata[1]}<br>"
                        "n_seeds=%{customdata[0]}<br>"
                        "metric=%{customdata[2]}<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )

        for feature_group, row_idx in [("agent_action0", 2), ("agent_non_idle", 3)]:
            metric_data = behavior[(behavior["family"] == family) & (behavior["feature_group"] == feature_group)].sort_values("step_millions")
            if metric_data.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=metric_data["step_millions"],
                    y=metric_data["mean_value_smooth"],
                    mode="lines",
                    name=label,
                    legendgroup=family,
                    showlegend=False,
                    line={"color": color, "width": 2.6},
                    customdata=np.stack([
                        metric_data["n_seeds"],
                        metric_data["seeds"].astype(str),
                        metric_data["metric_label"].astype(str),
                    ], axis=-1),
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        "step=%{x:.2f}M<br>"
                        "%{customdata[2]}=%{y:.4f}<br>"
                        "seeds=%{customdata[1]}<br>"
                        "n_seeds=%{customdata[0]}<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=1,
            )

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=940,
        width=1450,
        hovermode="x unified",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 340, "t": 95, "b": 70},
    )
    fig.update_yaxes(title_text="survival (%)", range=[0, 105], row=1, col=1)
    fig.update_yaxes(title_text="action 0 fraction", range=[0, 1], row=2, col=1)
    fig.update_yaxes(title_text="non-idle fraction", range=[0, 1], row=3, col=1)
    fig.update_xaxes(title_text="steps (M)", row=3, col=1)
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_survival_vs_action_ig15 = plot_survival_vs_action_behavior(
    "intervention_gate_15",
    "Intervention gate 15: survival vs action behavior over training",
    "survival_vs_action_behavior_intervention_gate_15",
)
fig_survival_vs_action_sparse16 = plot_survival_vs_action_behavior(
    "phase4_sparse_control_16",
    "Phase4 sparse control 16: survival vs action behavior over training",
    "survival_vs_action_behavior_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/survival_vs_action_behavior_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/survival_vs_action_behavior_phase4_sparse_control_16.html


## Entropy Collapse vs Action-0 Confidence

This compares action-0 usage with `train/entropy_agent_*`. High action-0 with high entropy suggests uncertainty/exploration; high action-0 with low entropy suggests a confident do-nothing policy.


In [32]:
ENTROPY_ACTION0_SMOOTH_WINDOW = 5
ENTROPY_ACTION0_FINAL_LAST_N_LOGGED = 5


def entropy_action0_run_rows():
    data = FEATURE_LONG[FEATURE_LONG["feature_group"].isin(["agent_action0", "agent_entropy"])].copy()
    if data.empty:
        return pd.DataFrame()

    group_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "feature_group",
        "_step",
        "step_millions",
    ]
    per_run = data.groupby(group_cols, dropna=False, as_index=False).agg(
        run_mean_value=("value", "mean"),
        run_std_agent_value=("value", "std"),
        n_agents_observed=("entity", "nunique"),
        agents=("entity", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    per_run["run_std_agent_value"] = per_run["run_std_agent_value"].fillna(0.0)
    per_run["metric_label"] = per_run["feature_group"].map({
        "agent_action0": "action 0 fraction",
        "agent_entropy": "policy entropy",
    })
    return per_run.sort_values(["experiment", "control_value", "design", "family_label", "feature_group", "_step"])


def seed_aggregated_entropy_action0_rows():
    if ENTROPY_ACTION0_RUN_LONG.empty:
        return pd.DataFrame()

    group_cols = [
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "feature_group",
        "metric_label",
        "_step",
        "step_millions",
    ]
    grouped = ENTROPY_ACTION0_RUN_LONG.groupby(group_cols, dropna=False, as_index=False).agg(
        mean_value=("run_mean_value", "mean"),
        std_value=("run_mean_value", "std"),
        mean_agent_spread=("run_std_agent_value", "mean"),
        min_agents_observed=("n_agents_observed", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_value"] = grouped["std_value"].fillna(0.0)
    grouped["mean_agent_spread"] = grouped["mean_agent_spread"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "feature_group", "_step"])

    if ENTROPY_ACTION0_SMOOTH_WINDOW and int(ENTROPY_ACTION0_SMOOTH_WINDOW) > 1:
        for column in ["mean_value", "std_value", "mean_agent_spread"]:
            grouped[f"{column}_smooth"] = grouped.groupby(
                ["experiment", "family", "feature_group"],
                dropna=False,
            )[column].transform(lambda values: values.rolling(int(ENTROPY_ACTION0_SMOOTH_WINDOW), min_periods=1).mean())
    else:
        for column in ["mean_value", "std_value", "mean_agent_spread"]:
            grouped[f"{column}_smooth"] = grouped[column]
    return grouped


def seed_aggregated_final_entropy_action0_rows():
    if ENTROPY_ACTION0_RUN_LONG.empty:
        return pd.DataFrame()

    data = ENTROPY_ACTION0_RUN_LONG.sort_values(["run_id", "feature_group", "_step"]).copy()
    data = data.groupby(["run_id", "feature_group"], dropna=False).tail(int(ENTROPY_ACTION0_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "feature_group",
    ]
    per_run = data.groupby(per_run_cols, dropna=False, as_index=False).agg(
        final_value=("run_mean_value", "mean"),
        final_agent_spread=("run_std_agent_value", "mean"),
        final_step_millions=("step_millions", "max"),
        min_agents_observed=("n_agents_observed", "min"),
        averaged_logged_points=("run_mean_value", "count"),
    )

    index_cols = [col for col in per_run_cols if col != "feature_group"]
    values = per_run.pivot_table(index=index_cols, columns="feature_group", values="final_value", aggfunc="mean").reset_index()
    spreads = per_run.pivot_table(index=index_cols, columns="feature_group", values="final_agent_spread", aggfunc="mean").reset_index()
    counts = per_run.groupby(index_cols, dropna=False, as_index=False).agg(
        final_step_millions=("final_step_millions", "max"),
        min_agents_observed=("min_agents_observed", "min"),
        min_averaged_logged_points=("averaged_logged_points", "min"),
    )

    for column in ["agent_action0", "agent_entropy"]:
        if column not in values.columns:
            values[column] = np.nan
        if column not in spreads.columns:
            spreads[column] = np.nan
    values = values.rename(columns={
        "agent_action0": "final_action0_fraction",
        "agent_entropy": "final_entropy",
    })
    spreads = spreads.rename(columns={
        "agent_action0": "final_action0_agent_spread",
        "agent_entropy": "final_entropy_agent_spread",
    })
    spread_cols = index_cols + ["final_action0_agent_spread", "final_entropy_agent_spread"]
    per_run_wide = values.merge(spreads[spread_cols], on=index_cols, how="left").merge(counts, on=index_cols, how="left")
    per_run_wide = per_run_wide.dropna(subset=["final_action0_fraction", "final_entropy"], how="any")
    if per_run_wide.empty:
        return pd.DataFrame()

    grouped = per_run_wide.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_final_action0_fraction=("final_action0_fraction", "mean"),
        std_final_action0_fraction=("final_action0_fraction", "std"),
        mean_final_entropy=("final_entropy", "mean"),
        std_final_entropy=("final_entropy", "std"),
        mean_final_action0_agent_spread=("final_action0_agent_spread", "mean"),
        mean_final_entropy_agent_spread=("final_entropy_agent_spread", "mean"),
        mean_final_step_millions=("final_step_millions", "mean"),
        min_agents_observed=("min_agents_observed", "min"),
        min_averaged_logged_points=("min_averaged_logged_points", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    for column in ["std_final_action0_fraction", "std_final_entropy"]:
        grouped[column] = grouped[column].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    return grouped.sort_values(["experiment", "control_value", "design", "family_label"])


ENTROPY_ACTION0_RUN_LONG = entropy_action0_run_rows()
ENTROPY_ACTION0_SEED_AGG = seed_aggregated_entropy_action0_rows()
ENTROPY_ACTION0_FINAL_SEED_AGG = seed_aggregated_final_entropy_action0_rows()

# print("Seed-aggregated action-0 vs entropy time series:")
# if ENTROPY_ACTION0_SEED_AGG.empty:
#     print("No action-0/entropy metrics found.")
# else:
#     display(ENTROPY_ACTION0_SEED_AGG.groupby(["experiment", "family_label", "metric_label"], dropna=False).agg(
#         points=("_step", "count"),
#         n_seeds=("n_seeds", "max"),
#         seeds=("seeds", "first"),
#         min_agents_observed=("min_agents_observed", "min"),
#     ).reset_index())

# print(f"Seed-aggregated final action-0 vs entropy, averaged over last {ENTROPY_ACTION0_FINAL_LAST_N_LOGGED} logged points:")
# if ENTROPY_ACTION0_FINAL_SEED_AGG.empty:
#     print("No final action-0/entropy summary available.")
# else:
#     display(ENTROPY_ACTION0_FINAL_SEED_AGG[[
#         "experiment",
#         "family_label",
#         "mean_final_action0_fraction",
#         "std_final_action0_fraction",
#         "mean_final_entropy",
#         "std_final_entropy",
#         "n_seeds",
#         "seeds",
#     ]])


def plot_entropy_action0_timeseries(experiment, title, save_name):
    data = ENTROPY_ACTION0_SEED_AGG[ENTROPY_ACTION0_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No action-0/entropy time-series data for {experiment}")
        return None

    conditions = data[["family", "condition_label", "control_value", "design"]].drop_duplicates()
    conditions = conditions.sort_values(["control_value", "design", "condition_label"])
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {row.family: palette[idx % len(palette)] for idx, row in enumerate(conditions.itertuples(index=False))}

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.075,
        subplot_titles=(
            "Mean action-0 fraction across agents",
            "Mean policy entropy across agents",
        ),
    )
    for _, condition in conditions.iterrows():
        family = condition["family"]
        label = condition["condition_label"]
        color = colors[family]
        for feature_group, row_idx in [("agent_action0", 1), ("agent_entropy", 2)]:
            metric_data = data[(data["family"] == family) & (data["feature_group"] == feature_group)].sort_values("step_millions")
            if metric_data.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=metric_data["step_millions"],
                    y=metric_data["mean_value_smooth"],
                    mode="lines",
                    name=label,
                    legendgroup=family,
                    showlegend=row_idx == 1,
                    line={"color": color, "width": 3 if row_idx == 1 else 2.8},
                    customdata=np.stack([
                        metric_data["std_value_smooth"],
                        metric_data["mean_agent_spread_smooth"],
                        metric_data["n_seeds"],
                        metric_data["seeds"].astype(str),
                        metric_data["metric_label"].astype(str),
                    ], axis=-1),
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        "step=%{x:.2f}M<br>"
                        "%{customdata[4]}=%{y:.4f}<br>"
                        "std across seeds=%{customdata[0]:.4f}<br>"
                        "mean agent spread=%{customdata[1]:.4f}<br>"
                        "seeds=%{customdata[3]}<br>"
                        "n_seeds=%{customdata[2]}<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=1,
            )

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=780,
        width=1450,
        hovermode="x unified",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 95, "b": 70},
    )
    fig.update_yaxes(title_text="action-0 fraction", range=[0, 1], row=1, col=1)
    fig.update_yaxes(title_text="entropy", row=2, col=1)
    fig.update_xaxes(title_text="steps (M)", row=2, col=1)
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


def plot_final_action0_entropy_scatter(experiment, title, save_name):
    data = ENTROPY_ACTION0_FINAL_SEED_AGG[ENTROPY_ACTION0_FINAL_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No final action-0/entropy data for {experiment}")
        return None

    data = data.sort_values(["control_value", "design", "condition_label"])
    fig = px.scatter(
        data,
        x="mean_final_action0_fraction",
        y="mean_final_entropy",
        color="condition_label",
        symbol="design",
        size="n_seeds",
        size_max=18,
        error_x="std_final_action0_fraction",
        error_y="std_final_entropy",
        hover_data={
            "condition_label": True,
            "control_label": True,
            "design": True,
            "mean_final_action0_fraction": ":.4f",
            "std_final_action0_fraction": ":.4f",
            "mean_final_entropy": ":.4f",
            "std_final_entropy": ":.4f",
            "mean_final_action0_agent_spread": ":.4f",
            "mean_final_entropy_agent_spread": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "min_averaged_logged_points": True,
        },
        labels={
            "mean_final_action0_fraction": "mean final action-0 fraction",
            "mean_final_entropy": "mean final entropy",
            "condition_label": "condition",
        },
        title=title,
    )
    fig.update_xaxes(range=[0, 1], title_text="action-0 fraction")
    fig.update_yaxes(title_text="policy entropy")
    fig.update_traces(marker={"line": {"width": 1, "color": "white"}})
    fig.update_layout(
        template="plotly_white",
        height=640,
        width=1450,
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 90, "b": 80},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_entropy_action0_timeseries_ig15 = plot_entropy_action0_timeseries(
    "intervention_gate_15",
    "Intervention gate 15: action-0 fraction vs entropy over training",
    "entropy_action0_timeseries_intervention_gate_15",
)
fig_entropy_action0_timeseries_sparse16 = plot_entropy_action0_timeseries(
    "phase4_sparse_control_16",
    "Phase4 sparse control 16: action-0 fraction vs entropy over training",
    "entropy_action0_timeseries_phase4_sparse_control_16",
)
fig_entropy_action0_final_scatter_ig15 = plot_final_action0_entropy_scatter(
    "intervention_gate_15",
    f"Intervention gate 15: final action-0 confidence map (mean of last {ENTROPY_ACTION0_FINAL_LAST_N_LOGGED} logged)",
    "final_entropy_action0_confidence_intervention_gate_15",
)
fig_entropy_action0_final_scatter_sparse16 = plot_final_action0_entropy_scatter(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: final action-0 confidence map (mean of last {ENTROPY_ACTION0_FINAL_LAST_N_LOGGED} logged)",
    "final_entropy_action0_confidence_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/entropy_action0_timeseries_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/entropy_action0_timeseries_phase4_sparse_control_16.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_entropy_action0_confidence_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_entropy_action0_confidence_phase4_sparse_control_16.html


## Agent Non-Idle Imbalance

This diagnostic measures whether control is shared across agents or concentrated in one subgraph: `max(non_idle_fraction_agent_i) - min(non_idle_fraction_agent_i)`.


In [33]:
AGENT_IMBALANCE_SMOOTH_WINDOW = 5
AGENT_IMBALANCE_FINAL_LAST_N_LOGGED = 5


def agent_imbalance_run_rows():
    data = FEATURE_LONG[FEATURE_LONG["feature_group"] == "agent_non_idle"].copy()
    if data.empty:
        return pd.DataFrame()

    group_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "_step",
        "step_millions",
    ]
    grouped = data.groupby(group_cols, dropna=False, as_index=False).agg(
        min_non_idle=("value", "min"),
        max_non_idle=("value", "max"),
        mean_non_idle=("value", "mean"),
        std_agent_non_idle=("value", "std"),
        n_agents_observed=("entity", "nunique"),
        agents=("entity", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_agent_non_idle"] = grouped["std_agent_non_idle"].fillna(0.0)
    grouped["non_idle_imbalance"] = grouped["max_non_idle"] - grouped["min_non_idle"]
    return grouped.sort_values(["experiment", "control_value", "design", "family_label", "_step"])


def seed_aggregated_agent_imbalance_rows():
    if AGENT_IMBALANCE_RUN_LONG.empty:
        return pd.DataFrame()

    group_cols = [
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "_step",
        "step_millions",
    ]
    grouped = AGENT_IMBALANCE_RUN_LONG.groupby(group_cols, dropna=False, as_index=False).agg(
        mean_imbalance=("non_idle_imbalance", "mean"),
        std_imbalance=("non_idle_imbalance", "std"),
        mean_non_idle=("mean_non_idle", "mean"),
        mean_min_non_idle=("min_non_idle", "mean"),
        mean_max_non_idle=("max_non_idle", "mean"),
        min_agents_observed=("n_agents_observed", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_imbalance"] = grouped["std_imbalance"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "_step"])

    if AGENT_IMBALANCE_SMOOTH_WINDOW and int(AGENT_IMBALANCE_SMOOTH_WINDOW) > 1:
        for column in ["mean_imbalance", "std_imbalance", "mean_non_idle", "mean_min_non_idle", "mean_max_non_idle"]:
            grouped[f"{column}_smooth"] = grouped.groupby(
                ["experiment", "family"],
                dropna=False,
            )[column].transform(lambda values: values.rolling(int(AGENT_IMBALANCE_SMOOTH_WINDOW), min_periods=1).mean())
    else:
        for column in ["mean_imbalance", "std_imbalance", "mean_non_idle", "mean_min_non_idle", "mean_max_non_idle"]:
            grouped[f"{column}_smooth"] = grouped[column]
    return grouped


def seed_aggregated_final_agent_imbalance_rows():
    if AGENT_IMBALANCE_RUN_LONG.empty:
        return pd.DataFrame()

    data = AGENT_IMBALANCE_RUN_LONG.sort_values(["run_id", "_step"]).copy()
    data = data.groupby(["run_id"], dropna=False).tail(int(AGENT_IMBALANCE_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
    ]
    per_run = data.groupby(per_run_cols, dropna=False, as_index=False).agg(
        final_imbalance=("non_idle_imbalance", "mean"),
        final_mean_non_idle=("mean_non_idle", "mean"),
        final_min_non_idle=("min_non_idle", "mean"),
        final_max_non_idle=("max_non_idle", "mean"),
        final_step_millions=("step_millions", "max"),
        min_agents_observed=("n_agents_observed", "min"),
        averaged_logged_points=("non_idle_imbalance", "count"),
    )

    grouped = per_run.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_final_imbalance=("final_imbalance", "mean"),
        std_final_imbalance=("final_imbalance", "std"),
        mean_final_non_idle=("final_mean_non_idle", "mean"),
        mean_final_min_non_idle=("final_min_non_idle", "mean"),
        mean_final_max_non_idle=("final_max_non_idle", "mean"),
        mean_final_step_millions=("final_step_millions", "mean"),
        min_agents_observed=("min_agents_observed", "min"),
        min_averaged_logged_points=("averaged_logged_points", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_final_imbalance"] = grouped["std_final_imbalance"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    return grouped.sort_values(["experiment", "control_value", "design", "family_label"])


AGENT_IMBALANCE_RUN_LONG = agent_imbalance_run_rows()
AGENT_IMBALANCE_SEED_AGG = seed_aggregated_agent_imbalance_rows()
AGENT_IMBALANCE_FINAL_SEED_AGG = seed_aggregated_final_agent_imbalance_rows()

# print("Seed-aggregated non-idle imbalance time series:")
# if AGENT_IMBALANCE_SEED_AGG.empty:
#     print("No agent non-idle metrics found for imbalance plots.")
# else:
#     display(AGENT_IMBALANCE_SEED_AGG.groupby(["experiment", "family_label"], dropna=False).agg(
#         points=("_step", "count"),
#         n_seeds=("n_seeds", "max"),
#         seeds=("seeds", "first"),
#         min_agents_observed=("min_agents_observed", "min"),
#     ).reset_index())

# print(f"Seed-aggregated final non-idle imbalance, averaged over last {AGENT_IMBALANCE_FINAL_LAST_N_LOGGED} logged points:")
# if AGENT_IMBALANCE_FINAL_SEED_AGG.empty:
#     print("No final agent imbalance summary available.")
# else:
#     display(AGENT_IMBALANCE_FINAL_SEED_AGG[[
#         "experiment",
#         "family_label",
#         "mean_final_imbalance",
#         "std_final_imbalance",
#         "mean_final_non_idle",
#         "mean_final_min_non_idle",
#         "mean_final_max_non_idle",
#         "n_seeds",
#         "seeds",
#     ]])


def plot_agent_imbalance_timeseries(experiment, title, save_name):
    data = AGENT_IMBALANCE_SEED_AGG[AGENT_IMBALANCE_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No agent imbalance time-series data for {experiment}")
        return None

    conditions = data[["family", "condition_label", "control_value", "design"]].drop_duplicates()
    conditions = conditions.sort_values(["control_value", "design", "condition_label"])
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {row.family: palette[idx % len(palette)] for idx, row in enumerate(conditions.itertuples(index=False))}

    fig = go.Figure()
    for _, condition in conditions.iterrows():
        family = condition["family"]
        label = condition["condition_label"]
        condition_data = data[data["family"] == family].sort_values("step_millions")
        if condition_data.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=condition_data["step_millions"],
                y=condition_data["mean_imbalance_smooth"],
                mode="lines",
                name=label,
                line={"color": colors[family], "width": 3},
                customdata=np.stack([
                    condition_data["std_imbalance_smooth"],
                    condition_data["mean_non_idle_smooth"],
                    condition_data["mean_min_non_idle_smooth"],
                    condition_data["mean_max_non_idle_smooth"],
                    condition_data["n_seeds"],
                    condition_data["seeds"].astype(str),
                ], axis=-1),
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    "step=%{x:.2f}M<br>"
                    "imbalance=%{y:.4f}<br>"
                    "std=%{customdata[0]:.4f}<br>"
                    "mean non-idle=%{customdata[1]:.4f}<br>"
                    "min agent=%{customdata[2]:.4f}<br>"
                    "max agent=%{customdata[3]:.4f}<br>"
                    "seeds=%{customdata[5]}<br>"
                    "n_seeds=%{customdata[4]}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=620,
        width=1450,
        hovermode="x unified",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 90, "b": 70},
    )
    fig.update_yaxes(title_text="max(non-idle) - min(non-idle)", range=[0, 1])
    fig.update_xaxes(title_text="steps (M)")
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


def _final_agent_imbalance_seed_points(experiment):
    data = AGENT_IMBALANCE_RUN_LONG[AGENT_IMBALANCE_RUN_LONG["experiment"] == experiment].copy()
    if data.empty:
        return pd.DataFrame()
    data = data.sort_values(["run_id", "_step"])
    data = data.groupby(["run_id"], dropna=False).tail(int(AGENT_IMBALANCE_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
    ]
    seed_points = data.groupby(per_run_cols, dropna=False, as_index=False).agg(
        seed_final_imbalance=("non_idle_imbalance", "mean"),
        seed_final_non_idle=("mean_non_idle", "mean"),
        seed_final_min_non_idle=("min_non_idle", "mean"),
        seed_final_max_non_idle=("max_non_idle", "mean"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("non_idle_imbalance", "count"),
    )
    seed_points["condition_label"] = seed_points["family_label"]
    return seed_points.sort_values(["control_value", "design", "condition_label", "seed"])


def plot_final_agent_imbalance(experiment, title, save_name):
    data = AGENT_IMBALANCE_FINAL_SEED_AGG[AGENT_IMBALANCE_FINAL_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No final agent imbalance data for {experiment}")
        return None

    data = data.sort_values(["control_value", "design", "condition_label"])
    fig = px.bar(
        data,
        x="condition_label",
        y="mean_final_imbalance",
        color="condition_label",
        error_y="std_final_imbalance",
        hover_data={
            "condition_label": True,
            "control_label": True,
            "design": True,
            "mean_final_imbalance": ":.4f",
            "std_final_imbalance": ":.4f",
            "mean_final_non_idle": ":.4f",
            "mean_final_min_non_idle": ":.4f",
            "mean_final_max_non_idle": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "min_averaged_logged_points": True,
        },
        labels={
            "condition_label": "condition",
            "mean_final_imbalance": "mean final imbalance",
        },
        title=title,
    )

    seed_points = _final_agent_imbalance_seed_points(experiment)
    add_seed_point_overlay(
        fig,
        seed_points,
        x_col="condition_label",
        y_col="seed_final_imbalance",
        group_col="condition_label",
        customdata_cols=[
            "run_name",
            "seed",
            "condition_label",
            "seed_final_non_idle",
            "seed_final_min_non_idle",
            "seed_final_max_non_idle",
            "averaged_logged_points",
        ],
        hovertemplate=(
            "seed run=%{customdata[0]}<br>"
            "seed=%{customdata[1]}<br>"
            "condition=%{customdata[2]}<br>"
            "seed imbalance=%{y:.4f}<br>"
            "mean non-idle=%{customdata[3]:.4f}<br>"
            "min agent=%{customdata[4]:.4f}<br>"
            "max agent=%{customdata[5]:.4f}<br>"
            "logged points=%{customdata[6]}<extra></extra>"
        ),
    )

    fig.update_yaxes(title_text="max(non-idle) - min(non-idle)", range=[0, 1])
    fig.update_xaxes(tickangle=25)
    fig.update_layout(
        template="plotly_white",
        height=640,
        width=1450,
        showlegend=False,
        bargap=0.22,
        margin={"l": 80, "r": 80, "t": 90, "b": 150},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_agent_imbalance_timeseries_ig15 = plot_agent_imbalance_timeseries(
    "intervention_gate_15",
    "Intervention gate 15: agent non-idle imbalance over training",
    "agent_non_idle_imbalance_timeseries_intervention_gate_15",
)
fig_agent_imbalance_timeseries_sparse16 = plot_agent_imbalance_timeseries(
    "phase4_sparse_control_16",
    "Phase4 sparse control 16: agent non-idle imbalance over training",
    "agent_non_idle_imbalance_timeseries_phase4_sparse_control_16",
)
fig_agent_imbalance_final_ig15 = plot_final_agent_imbalance(
    "intervention_gate_15",
    f"Intervention gate 15: final agent non-idle imbalance (mean of last {AGENT_IMBALANCE_FINAL_LAST_N_LOGGED} logged)",
    "final_agent_non_idle_imbalance_intervention_gate_15",
)
fig_agent_imbalance_final_sparse16 = plot_final_agent_imbalance(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: final agent non-idle imbalance (mean of last {AGENT_IMBALANCE_FINAL_LAST_N_LOGGED} logged)",
    "final_agent_non_idle_imbalance_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/agent_non_idle_imbalance_timeseries_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/agent_non_idle_imbalance_timeseries_phase4_sparse_control_16.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_agent_non_idle_imbalance_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_agent_non_idle_imbalance_phase4_sparse_control_16.html


## Joint Action Coordination

These plots compare how often 0, 1, 2, or 3 agents act at the same environment step. This separates useful sparse coordination from the trivial case where every agent simply stays idle.


In [34]:
JOINT_COORDINATION_SMOOTH_WINDOW = 5
JOINT_COORDINATION_FINAL_LAST_N_LOGGED = 5
JOINT_AGENT_COUNT_LABELS = {
    0: "0 agents act",
    1: "1 agent acts",
    2: "2 agents act",
    3: "3 agents act",
}
JOINT_AGENT_COUNT_ORDER = list(JOINT_AGENT_COUNT_LABELS.values())


def joint_coordination_run_rows():
    data = FEATURE_LONG[FEATURE_LONG["feature_group"] == "joint_non_idle"].copy()
    if data.empty:
        return pd.DataFrame()

    data["n_agents_act"] = pd.to_numeric(data["entity"], errors="coerce")
    data = data.dropna(subset=["n_agents_act", "value", "_step"])
    data["n_agents_act"] = data["n_agents_act"].astype(int)
    data = data[data["n_agents_act"].isin(JOINT_AGENT_COUNT_LABELS.keys())].copy()
    data["coordination_label"] = data["n_agents_act"].map(JOINT_AGENT_COUNT_LABELS)
    data["coordination_label"] = pd.Categorical(
        data["coordination_label"],
        categories=JOINT_AGENT_COUNT_ORDER,
        ordered=True,
    )
    return data.sort_values(["experiment", "control_value", "design", "family_label", "n_agents_act", "_step"])


def seed_aggregated_joint_coordination_rows():
    if JOINT_COORDINATION_RUN_LONG.empty:
        return pd.DataFrame()

    group_cols = [
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "n_agents_act",
        "coordination_label",
        "_step",
        "step_millions",
    ]
    grouped = JOINT_COORDINATION_RUN_LONG.groupby(
        group_cols,
        dropna=False,
        observed=True,
        as_index=False,
    ).agg(
        mean_fraction=("value", "mean"),
        std_fraction=("value", "std"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_fraction"] = grouped["std_fraction"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped["coordination_label"] = pd.Categorical(
        grouped["coordination_label"],
        categories=JOINT_AGENT_COUNT_ORDER,
        ordered=True,
    )
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "n_agents_act", "_step"])

    if JOINT_COORDINATION_SMOOTH_WINDOW and int(JOINT_COORDINATION_SMOOTH_WINDOW) > 1:
        grouped["mean_fraction_smooth"] = grouped.groupby(
            ["experiment", "family", "n_agents_act"],
            dropna=False,
            observed=True,
        )["mean_fraction"].transform(lambda values: values.rolling(int(JOINT_COORDINATION_SMOOTH_WINDOW), min_periods=1).mean())
        grouped["std_fraction_smooth"] = grouped.groupby(
            ["experiment", "family", "n_agents_act"],
            dropna=False,
            observed=True,
        )["std_fraction"].transform(lambda values: values.rolling(int(JOINT_COORDINATION_SMOOTH_WINDOW), min_periods=1).mean())
    else:
        grouped["mean_fraction_smooth"] = grouped["mean_fraction"]
        grouped["std_fraction_smooth"] = grouped["std_fraction"]
    return grouped


def seed_aggregated_final_joint_coordination_rows():
    if JOINT_COORDINATION_RUN_LONG.empty:
        return pd.DataFrame()

    data = JOINT_COORDINATION_RUN_LONG.sort_values(["run_id", "n_agents_act", "_step"]).copy()
    data = data.groupby(["run_id", "n_agents_act"], dropna=False, observed=True).tail(int(JOINT_COORDINATION_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "n_agents_act",
        "coordination_label",
    ]
    per_run = data.groupby(per_run_cols, dropna=False, observed=True, as_index=False).agg(
        final_fraction=("value", "mean"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("value", "count"),
    )

    grouped = per_run.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "n_agents_act",
            "coordination_label",
        ],
        dropna=False,
        observed=True,
        as_index=False,
    ).agg(
        mean_final_fraction=("final_fraction", "mean"),
        std_final_fraction=("final_fraction", "std"),
        mean_final_step_millions=("final_step_millions", "mean"),
        min_averaged_logged_points=("averaged_logged_points", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    grouped["std_final_fraction"] = grouped["std_final_fraction"].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped["coordination_label"] = pd.Categorical(
        grouped["coordination_label"],
        categories=JOINT_AGENT_COUNT_ORDER,
        ordered=True,
    )
    return grouped.sort_values(["experiment", "control_value", "design", "family_label", "n_agents_act"])


def joint_coordination_coverage_rows():
    keep_cols = [
        "experiment",
        "run_name",
        "run_id",
        "family",
        "family_label",
        "seed",
        "design",
        "control_label",
    ]
    if selected_runs.empty:
        return pd.DataFrame(columns=keep_cols + ["has_joint_coordination", "joint_coordination_rows", "joint_coordination_bins"])

    if JOINT_COORDINATION_RUN_LONG.empty:
        available = pd.DataFrame(columns=["run_id", "run_name", "joint_coordination_rows", "joint_coordination_bins"])
    else:
        available = JOINT_COORDINATION_RUN_LONG.groupby(["run_id", "run_name"], dropna=False, as_index=False).agg(
            joint_coordination_rows=("value", "count"),
            joint_coordination_bins=("coordination_label", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            first_step_millions=("step_millions", "min"),
            last_step_millions=("step_millions", "max"),
        )
    out = selected_runs[keep_cols].merge(available, on=["run_id", "run_name"], how="left")
    out["joint_coordination_rows"] = out["joint_coordination_rows"].fillna(0).astype(int)
    out["has_joint_coordination"] = out["joint_coordination_rows"] > 0
    return out.sort_values(["experiment", "family_label", "seed"])


JOINT_COORDINATION_RUN_LONG = joint_coordination_run_rows()
JOINT_COORDINATION_SEED_AGG = seed_aggregated_joint_coordination_rows()
JOINT_COORDINATION_FINAL_SEED_AGG = seed_aggregated_final_joint_coordination_rows()
JOINT_COORDINATION_COVERAGE = joint_coordination_coverage_rows()

# print("Joint coordination metric coverage for cached runs:")
# if JOINT_COORDINATION_COVERAGE.empty:
#     print("No cached runs found in the selected cache.")
# else:
#     display(JOINT_COORDINATION_COVERAGE[[
#         "experiment",
#         "run_name",
#         "family_label",
#         "seed",
#         "has_joint_coordination",
#         "joint_coordination_rows",
#         "joint_coordination_bins",
#     ]])

# missing_joint_coordination = JOINT_COORDINATION_COVERAGE[~JOINT_COORDINATION_COVERAGE["has_joint_coordination"]]
# if not missing_joint_coordination.empty:
#     print("Cached runs with no non-empty joint coordination rows:")
#     display(missing_joint_coordination[["experiment", "run_name", "family_label", "seed"]])

# print("Seed-aggregated joint action coordination time series:")
# if JOINT_COORDINATION_SEED_AGG.empty:
#     print("No joint non-idle count metrics found.")
# else:
#     display(JOINT_COORDINATION_SEED_AGG.groupby(["experiment", "family_label", "coordination_label"], dropna=False, observed=True).agg(
#         points=("_step", "count"),
#         n_seeds=("n_seeds", "max"),
#         seeds=("seeds", "first"),
#     ).reset_index())

# print(f"Seed-aggregated final joint action coordination, averaged over last {JOINT_COORDINATION_FINAL_LAST_N_LOGGED} logged points:")
# if JOINT_COORDINATION_FINAL_SEED_AGG.empty:
#     print("No final joint coordination summary available.")
# else:
#     display(JOINT_COORDINATION_FINAL_SEED_AGG[[
#         "experiment",
#         "family_label",
#         "coordination_label",
#         "mean_final_fraction",
#         "std_final_fraction",
#         "n_seeds",
#         "seeds",
#     ]])


def plot_joint_coordination_timeseries(experiment, title, save_name):
    data = JOINT_COORDINATION_SEED_AGG[JOINT_COORDINATION_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No joint coordination time-series data for {experiment}")
        return None

    counts = [count for count in JOINT_AGENT_COUNT_LABELS if count in set(data["n_agents_act"].astype(int))]
    conditions = data[["family", "condition_label", "control_value", "design"]].drop_duplicates()
    conditions = conditions.sort_values(["control_value", "design", "condition_label"])
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {row.family: palette[idx % len(palette)] for idx, row in enumerate(conditions.itertuples(index=False))}

    fig = make_subplots(
        rows=len(counts),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.055,
        subplot_titles=[JOINT_AGENT_COUNT_LABELS[count] for count in counts],
    )

    for row_idx, count in enumerate(counts, start=1):
        count_data = data[data["n_agents_act"].astype(int) == int(count)]
        for _, condition in conditions.iterrows():
            family = condition["family"]
            label = condition["condition_label"]
            condition_data = count_data[count_data["family"] == family].sort_values("step_millions")
            if condition_data.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=condition_data["step_millions"],
                    y=condition_data["mean_fraction_smooth"],
                    mode="lines",
                    name=label,
                    legendgroup=family,
                    showlegend=row_idx == 1,
                    line={"color": colors[family], "width": 2.8},
                    customdata=np.stack([
                        condition_data["n_seeds"],
                        condition_data["seeds"].astype(str),
                        condition_data["std_fraction_smooth"],
                    ], axis=-1),
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        f"coordination={JOINT_AGENT_COUNT_LABELS[count]}<br>"
                        "step=%{x:.2f}M<br>"
                        "fraction=%{y:.4f}<br>"
                        "std=%{customdata[2]:.4f}<br>"
                        "seeds=%{customdata[1]}<br>"
                        "n_seeds=%{customdata[0]}<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=1,
            )
        fig.update_yaxes(title_text="fraction", range=[0, 1], row=row_idx, col=1)

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=1080,
        width=1450,
        hovermode="x unified",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 100, "b": 70},
    )
    fig.update_xaxes(title_text="steps (M)", row=len(counts), col=1)
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


def _final_joint_coordination_seed_points(experiment):
    data = JOINT_COORDINATION_RUN_LONG[JOINT_COORDINATION_RUN_LONG["experiment"] == experiment].copy()
    if data.empty:
        return pd.DataFrame()
    data = data.sort_values(["run_id", "n_agents_act", "_step"])
    data = data.groupby(["run_id", "n_agents_act"], dropna=False, observed=True).tail(int(JOINT_COORDINATION_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "n_agents_act",
        "coordination_label",
    ]
    seed_points = data.groupby(per_run_cols, dropna=False, observed=True, as_index=False).agg(
        seed_final_fraction=("value", "mean"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("value", "count"),
    )
    seed_points["condition_label"] = seed_points["family_label"]
    seed_points["coordination_label"] = pd.Categorical(
        seed_points["coordination_label"],
        categories=JOINT_AGENT_COUNT_ORDER,
        ordered=True,
    )
    return seed_points.sort_values(["n_agents_act", "control_value", "design", "condition_label", "seed"])


def plot_final_joint_coordination(experiment, title, save_name):
    data = JOINT_COORDINATION_FINAL_SEED_AGG[JOINT_COORDINATION_FINAL_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No final joint coordination data for {experiment}")
        return None

    data["coordination_label"] = pd.Categorical(
        data["coordination_label"],
        categories=JOINT_AGENT_COUNT_ORDER,
        ordered=True,
    )
    fig = px.bar(
        data.sort_values(["n_agents_act", "control_value", "design", "condition_label"]),
        x="coordination_label",
        y="mean_final_fraction",
        color="condition_label",
        error_y="std_final_fraction",
        barmode="group",
        hover_data={
            "condition_label": True,
            "control_label": True,
            "design": True,
            "mean_final_fraction": ":.4f",
            "std_final_fraction": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "min_averaged_logged_points": True,
        },
        labels={
            "coordination_label": "agents acting in same step",
            "mean_final_fraction": "mean final fraction",
            "condition_label": "condition",
        },
        title=title,
    )

    seed_points = _final_joint_coordination_seed_points(experiment)
    add_seed_point_overlay(
        fig,
        seed_points,
        x_col="coordination_label",
        y_col="seed_final_fraction",
        group_col="condition_label",
        customdata_cols=[
            "run_name",
            "seed",
            "condition_label",
            "coordination_label",
            "seed_final_fraction",
            "averaged_logged_points",
        ],
        hovertemplate=(
            "seed run=%{customdata[0]}<br>"
            "seed=%{customdata[1]}<br>"
            "condition=%{customdata[2]}<br>"
            "coordination=%{customdata[3]}<br>"
            "seed fraction=%{y:.4f}<br>"
            "logged points=%{customdata[5]}<extra></extra>"
        ),
    )

    fig.update_yaxes(title_text="fraction of environment steps", range=[0, 1])
    fig.update_layout(
        template="plotly_white",
        height=640,
        width=1450,
        bargap=0.18,
        bargroupgap=0.04,
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 90, "b": 80},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_joint_coordination_timeseries_ig15 = plot_joint_coordination_timeseries(
    "intervention_gate_15",
    "Intervention gate 15: joint action coordination over training",
    "joint_action_coordination_timeseries_intervention_gate_15",
)
fig_joint_coordination_timeseries_sparse16 = plot_joint_coordination_timeseries(
    "phase4_sparse_control_16",
    "Phase4 sparse control 16: joint action coordination over training",
    "joint_action_coordination_timeseries_phase4_sparse_control_16",
)
fig_joint_coordination_final_ig15 = plot_final_joint_coordination(
    "intervention_gate_15",
    f"Intervention gate 15: final joint action coordination (mean of last {JOINT_COORDINATION_FINAL_LAST_N_LOGGED} logged)",
    "final_joint_action_coordination_intervention_gate_15",
)
fig_joint_coordination_final_sparse16 = plot_final_joint_coordination(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: final joint action coordination (mean of last {JOINT_COORDINATION_FINAL_LAST_N_LOGGED} logged)",
    "final_joint_action_coordination_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/joint_action_coordination_timeseries_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/joint_action_coordination_timeseries_phase4_sparse_control_16.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_joint_action_coordination_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_joint_action_coordination_phase4_sparse_control_16.html


## Gate Probability vs Actual Intervention Fraction


In [35]:
GATE_CALIBRATION_SMOOTH_WINDOW = 5
GATE_CALIBRATION_FINAL_LAST_N_LOGGED = 5


def gate_calibration_run_rows():
    gate = FEATURE_LONG[FEATURE_LONG["feature_group"].isin(["gate_prob_intervene", "gate_intervene_frac"])].copy()
    if gate.empty:
        return pd.DataFrame()

    gate = gate[
        gate["design"].astype(str).eq("gated")
        | gate["gate_enabled"].map(lambda value: value is True).fillna(False)
    ].copy()
    if gate.empty:
        return pd.DataFrame()

    index_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "gate_enabled",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "entity",
        "_step",
        "step_millions",
    ]
    wide = gate.pivot_table(
        index=index_cols,
        columns="feature_group",
        values="value",
        aggfunc="mean",
    ).reset_index()
    wide.columns.name = None
    for column in ["gate_prob_intervene", "gate_intervene_frac"]:
        if column not in wide.columns:
            wide[column] = np.nan
    wide = wide.dropna(subset=["gate_prob_intervene", "gate_intervene_frac"], how="all")
    if wide.empty:
        return pd.DataFrame()

    wide["prob_minus_actual"] = wide["gate_prob_intervene"] - wide["gate_intervene_frac"]
    wide["actual_minus_prob"] = wide["gate_intervene_frac"] - wide["gate_prob_intervene"]
    wide["agent"] = wide["entity"].astype(str)
    wide["agent"] = pd.Categorical(wide["agent"], categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    return wide.sort_values(["experiment", "control_value", "design", "family_label", "agent", "_step"])


def seed_aggregated_gate_calibration_rows():
    if GATE_CALIBRATION_RUN_LONG.empty:
        return pd.DataFrame()

    group_cols = [
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "entity",
        "agent",
        "_step",
        "step_millions",
    ]
    grouped = GATE_CALIBRATION_RUN_LONG.groupby(
        group_cols,
        dropna=False,
        observed=True,
        as_index=False,
    ).agg(
        mean_gate_prob=("gate_prob_intervene", "mean"),
        std_gate_prob=("gate_prob_intervene", "std"),
        mean_intervene_frac=("gate_intervene_frac", "mean"),
        std_intervene_frac=("gate_intervene_frac", "std"),
        mean_prob_minus_actual=("prob_minus_actual", "mean"),
        std_prob_minus_actual=("prob_minus_actual", "std"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    for column in ["std_gate_prob", "std_intervene_frac", "std_prob_minus_actual"]:
        grouped[column] = grouped[column].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped["agent"] = pd.Categorical(grouped["agent"].astype(str), categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    grouped = grouped.sort_values(["experiment", "control_value", "design", "family_label", "agent", "_step"])

    smooth_cols = [
        "mean_gate_prob",
        "std_gate_prob",
        "mean_intervene_frac",
        "std_intervene_frac",
        "mean_prob_minus_actual",
        "std_prob_minus_actual",
    ]
    if GATE_CALIBRATION_SMOOTH_WINDOW and int(GATE_CALIBRATION_SMOOTH_WINDOW) > 1:
        for column in smooth_cols:
            grouped[f"{column}_smooth"] = grouped.groupby(
                ["experiment", "family", "entity"],
                dropna=False,
                observed=True,
            )[column].transform(lambda values: values.rolling(int(GATE_CALIBRATION_SMOOTH_WINDOW), min_periods=1).mean())
    else:
        for column in smooth_cols:
            grouped[f"{column}_smooth"] = grouped[column]
    return grouped


def seed_aggregated_final_gate_calibration_rows():
    if GATE_CALIBRATION_RUN_LONG.empty:
        return pd.DataFrame()

    data = GATE_CALIBRATION_RUN_LONG.sort_values(["run_id", "agent", "_step"]).copy()
    data = data.groupby(["run_id", "agent"], dropna=False, observed=True).tail(int(GATE_CALIBRATION_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "entity",
        "agent",
    ]
    per_run = data.groupby(per_run_cols, dropna=False, observed=True, as_index=False).agg(
        final_gate_prob=("gate_prob_intervene", "mean"),
        final_intervene_frac=("gate_intervene_frac", "mean"),
        final_prob_minus_actual=("prob_minus_actual", "mean"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("prob_minus_actual", "count"),
    )

    grouped = per_run.groupby(
        [
            "experiment",
            "comparison_group",
            "family",
            "family_label",
            "design",
            "control_axis",
            "control_value",
            "control_label",
            "entropy_schedule",
            "entity",
            "agent",
        ],
        dropna=False,
        observed=True,
        as_index=False,
    ).agg(
        mean_final_gate_prob=("final_gate_prob", "mean"),
        std_final_gate_prob=("final_gate_prob", "std"),
        mean_final_intervene_frac=("final_intervene_frac", "mean"),
        std_final_intervene_frac=("final_intervene_frac", "std"),
        mean_final_prob_minus_actual=("final_prob_minus_actual", "mean"),
        std_final_prob_minus_actual=("final_prob_minus_actual", "std"),
        mean_final_step_millions=("final_step_millions", "mean"),
        min_averaged_logged_points=("averaged_logged_points", "min"),
        n_seeds=("run_id", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
    )
    for column in ["std_final_gate_prob", "std_final_intervene_frac", "std_final_prob_minus_actual"]:
        grouped[column] = grouped[column].fillna(0.0)
    grouped["condition_label"] = grouped["family_label"]
    grouped["agent"] = pd.Categorical(grouped["agent"].astype(str), categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    return grouped.sort_values(["experiment", "control_value", "design", "family_label", "agent"])


def gate_calibration_coverage_rows():
    gated_runs = selected_runs[
        selected_runs["design"].astype(str).eq("gated")
        | selected_runs["gate_enabled"].map(lambda value: value is True).fillna(False)
    ].copy()
    keep_cols = [
        "experiment",
        "run_name",
        "run_id",
        "family",
        "family_label",
        "seed",
        "design",
        "gate_enabled",
        "control_label",
    ]
    if gated_runs.empty:
        return pd.DataFrame(columns=keep_cols + ["has_gate_calibration", "gate_calibration_rows", "gate_agents"])

    if GATE_CALIBRATION_RUN_LONG.empty:
        available = pd.DataFrame(columns=["run_id", "run_name", "gate_calibration_rows", "gate_agents", "first_step_millions", "last_step_millions"])
    else:
        available = GATE_CALIBRATION_RUN_LONG.groupby(["run_id", "run_name"], dropna=False, as_index=False).agg(
            gate_calibration_rows=("prob_minus_actual", "count"),
            gate_agents=("agent", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            first_step_millions=("step_millions", "min"),
            last_step_millions=("step_millions", "max"),
        )
    out = gated_runs[keep_cols].merge(available, on=["run_id", "run_name"], how="left")
    out["gate_calibration_rows"] = out["gate_calibration_rows"].fillna(0).astype(int)
    out["has_gate_calibration"] = out["gate_calibration_rows"] > 0
    return out.sort_values(["experiment", "family_label", "seed"])


GATE_CALIBRATION_RUN_LONG = gate_calibration_run_rows()
GATE_CALIBRATION_SEED_AGG = seed_aggregated_gate_calibration_rows()
GATE_CALIBRATION_FINAL_SEED_AGG = seed_aggregated_final_gate_calibration_rows()
GATE_CALIBRATION_COVERAGE = gate_calibration_coverage_rows()

# print("Gate calibration coverage for gated runs:")
# if GATE_CALIBRATION_COVERAGE.empty:
#     print("No gated runs found in the selected cache.")
# else:
#     display(GATE_CALIBRATION_COVERAGE[[
#         "experiment",
#         "run_name",
#         "family_label",
#         "seed",
#         "has_gate_calibration",
#         "gate_calibration_rows",
#         "gate_agents",
#     ]])

missing_gate_calibration = GATE_CALIBRATION_COVERAGE[~GATE_CALIBRATION_COVERAGE["has_gate_calibration"]]
# if not missing_gate_calibration.empty:
#     print("Gated cached runs with no non-empty gate probability/intervention-fraction rows:")
#     display(missing_gate_calibration[["experiment", "run_name", "family_label", "seed"]])

# print("Seed-aggregated gate probability vs actual intervention time series:")
# if GATE_CALIBRATION_SEED_AGG.empty:
#     print("No gated intervention probability/fraction metrics found.")
# else:
#     display(GATE_CALIBRATION_SEED_AGG.groupby(["experiment", "family_label", "agent"], dropna=False, observed=True).agg(
#         points=("_step", "count"),
#         n_seeds=("n_seeds", "max"),
#         seeds=("seeds", "first"),
#     ).reset_index())

# print(f"Gate calibration final summary, averaged over last {GATE_CALIBRATION_FINAL_LAST_N_LOGGED} logged points:")
# if GATE_CALIBRATION_FINAL_SEED_AGG.empty:
#     print("No final gate calibration summary available.")
# else:
#     display(GATE_CALIBRATION_FINAL_SEED_AGG[[
#         "experiment",
#         "family_label",
#         "agent",
#         "mean_final_gate_prob",
#         "mean_final_intervene_frac",
#         "mean_final_prob_minus_actual",
#         "std_final_prob_minus_actual",
#         "n_seeds",
#         "seeds",
#     ]])


def plot_gate_probability_vs_intervention_timeseries(experiment, title, save_name):
    data = GATE_CALIBRATION_SEED_AGG[GATE_CALIBRATION_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No gate calibration data for {experiment}")
        return None

    agents = [agent for agent in ["agent_0", "agent_1", "agent_2"] if agent in set(data["agent"].astype(str))]
    if not agents:
        print(f"No agents found in gate calibration data for {experiment}")
        return None

    conditions = data[["family", "condition_label", "control_value", "design"]].drop_duplicates()
    conditions = conditions.sort_values(["control_value", "design", "condition_label"])
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {row.family: palette[idx % len(palette)] for idx, row in enumerate(conditions.itertuples(index=False))}

    fig = make_subplots(
        rows=len(agents),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.075,
        subplot_titles=[f"{agent}: predicted gate probability vs actual intervention fraction" for agent in agents],
    )

    for row_idx, agent in enumerate(agents, start=1):
        agent_data = data[data["agent"].astype(str) == agent]
        for _, condition in conditions.iterrows():
            family = condition["family"]
            label = condition["condition_label"]
            condition_data = agent_data[agent_data["family"] == family].sort_values("step_millions")
            if condition_data.empty:
                continue
            customdata = np.stack([
                condition_data["n_seeds"],
                condition_data["seeds"].astype(str),
                condition_data["mean_prob_minus_actual_smooth"],
            ], axis=-1)
            fig.add_trace(
                go.Scatter(
                    x=condition_data["step_millions"],
                    y=condition_data["mean_gate_prob_smooth"],
                    mode="lines",
                    name=f"{label} predicted",
                    legendgroup=f"{family}-predicted",
                    showlegend=row_idx == 1,
                    line={"color": colors[family], "width": 2.8},
                    customdata=customdata,
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        f"agent={agent}<br>"
                        "step=%{x:.2f}M<br>"
                        "gate prob intervene=%{y:.4f}<br>"
                        "predicted - actual=%{customdata[2]:.4f}<br>"
                        "seeds=%{customdata[1]}<br>"
                        "n_seeds=%{customdata[0]}<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=condition_data["step_millions"],
                    y=condition_data["mean_intervene_frac_smooth"],
                    mode="lines",
                    name=f"{label} actual",
                    legendgroup=f"{family}-actual",
                    showlegend=row_idx == 1,
                    line={"color": colors[family], "width": 2.8, "dash": "dash"},
                    customdata=customdata,
                    hovertemplate=(
                        f"<b>{label}</b><br>"
                        f"agent={agent}<br>"
                        "step=%{x:.2f}M<br>"
                        "actual intervene frac=%{y:.4f}<br>"
                        "predicted - actual=%{customdata[2]:.4f}<br>"
                        "seeds=%{customdata[1]}<br>"
                        "n_seeds=%{customdata[0]}<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=1,
            )
        fig.update_yaxes(title_text="fraction", range=[0, 1], row=row_idx, col=1)

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=880,
        width=1450,
        hovermode="x unified",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 380, "t": 100, "b": 70},
    )
    fig.update_xaxes(title_text="steps (M)", row=len(agents), col=1)
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


def _final_gate_calibration_seed_points(experiment):
    data = GATE_CALIBRATION_RUN_LONG[GATE_CALIBRATION_RUN_LONG["experiment"] == experiment].copy()
    if data.empty:
        return pd.DataFrame()
    data = data.sort_values(["run_id", "agent", "_step"])
    data = data.groupby(["run_id", "agent"], dropna=False, observed=True).tail(int(GATE_CALIBRATION_FINAL_LAST_N_LOGGED))
    per_run_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
        "entity",
        "agent",
    ]
    seed_points = data.groupby(per_run_cols, dropna=False, observed=True, as_index=False).agg(
        seed_final_gate_prob=("gate_prob_intervene", "mean"),
        seed_final_intervene_frac=("gate_intervene_frac", "mean"),
        seed_final_prob_minus_actual=("prob_minus_actual", "mean"),
        final_step_millions=("step_millions", "max"),
        averaged_logged_points=("prob_minus_actual", "count"),
    )
    seed_points["condition_label"] = seed_points["family_label"]
    seed_points["agent"] = pd.Categorical(seed_points["agent"].astype(str), categories=["agent_0", "agent_1", "agent_2"], ordered=True)
    return seed_points.sort_values(["control_value", "design", "condition_label", "agent", "seed"])


def plot_gate_calibration_final_gap(experiment, title, save_name):
    data = GATE_CALIBRATION_FINAL_SEED_AGG[GATE_CALIBRATION_FINAL_SEED_AGG["experiment"] == experiment].copy()
    if data.empty:
        print(f"No final gate calibration data for {experiment}")
        return None

    fig = px.bar(
        data,
        x="agent",
        y="mean_final_prob_minus_actual",
        color="condition_label",
        error_y="std_final_prob_minus_actual",
        barmode="group",
        hover_data={
            "condition_label": True,
            "control_label": True,
            "mean_final_gate_prob": ":.4f",
            "mean_final_intervene_frac": ":.4f",
            "mean_final_prob_minus_actual": ":.4f",
            "std_final_prob_minus_actual": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "min_averaged_logged_points": True,
        },
        labels={
            "agent": "agent",
            "mean_final_prob_minus_actual": "predicted - actual intervention fraction",
            "condition_label": "condition",
        },
        title=title,
    )

    seed_points = _final_gate_calibration_seed_points(experiment)
    add_seed_point_overlay(
        fig,
        seed_points,
        x_col="agent",
        y_col="seed_final_prob_minus_actual",
        group_col="condition_label",
        customdata_cols=[
            "run_name",
            "seed",
            "condition_label",
            "seed_final_gate_prob",
            "seed_final_intervene_frac",
            "averaged_logged_points",
        ],
        hovertemplate=(
            "seed run=%{customdata[0]}<br>"
            "seed=%{customdata[1]}<br>"
            "condition=%{customdata[2]}<br>"
            "agent=%{x}<br>"
            "seed predicted - actual=%{y:.4f}<br>"
            "gate prob=%{customdata[3]:.4f}<br>"
            "actual frac=%{customdata[4]:.4f}<br>"
            "logged points=%{customdata[5]}<extra></extra>"
        ),
    )

    fig.add_hline(y=0, line_width=1, line_color="black")
    fig.update_layout(
        template="plotly_white",
        height=620,
        width=1450,
        bargap=0.18,
        bargroupgap=0.04,
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 360, "t": 90, "b": 70},
    )
    fig.update_yaxes(title_text="predicted - actual intervention fraction")
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_gate_calibration_timeseries_ig15 = plot_gate_probability_vs_intervention_timeseries(
    "intervention_gate_15",
    "Intervention gate 15: gate probability vs actual intervention fraction over training",
    "gate_probability_vs_actual_intervention_timeseries_intervention_gate_15",
)
fig_gate_calibration_timeseries_sparse16 = plot_gate_probability_vs_intervention_timeseries(
    "phase4_sparse_control_16",
    "Phase4 sparse control 16: gate probability vs actual intervention fraction over training",
    "gate_probability_vs_actual_intervention_timeseries_phase4_sparse_control_16",
)
fig_gate_calibration_final_gap_ig15 = plot_gate_calibration_final_gap(
    "intervention_gate_15",
    f"Intervention gate 15: final gate calibration gap by agent (mean of last {GATE_CALIBRATION_FINAL_LAST_N_LOGGED} logged)",
    "final_gate_calibration_gap_intervention_gate_15",
)
fig_gate_calibration_final_gap_sparse16 = plot_gate_calibration_final_gap(
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: final gate calibration gap by agent (mean of last {GATE_CALIBRATION_FINAL_LAST_N_LOGGED} logged)",
    "final_gate_calibration_gap_phase4_sparse_control_16",
)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/gate_probability_vs_actual_intervention_timeseries_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/gate_probability_vs_actual_intervention_timeseries_phase4_sparse_control_16.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_gate_calibration_gap_intervention_gate_15.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/final_gate_calibration_gap_phase4_sparse_control_16.html


## Export Tables

In [36]:
tables_to_export = {
    "ig15_sparse16_expected_coverage": coverage,
    "ig15_sparse16_metric_availability": metric_availability,
    "ig15_sparse16_action_profile": ACTION_PROFILE,
    "ig15_sparse16_comparison_specs": COMPARISON_SPECS,
    "ig15_sparse16_comparison_deltas": COMPARISON_DELTAS,
    "ig15_sparse16_largest_changes": largest_changes,
    "ig15_sparse16_action0_count_summary": ACTION0_COUNT_SUMMARY,
    "ig15_sparse16_seed_aggregated_action0": ACTION0_SEED_AGG,
    "ig15_sparse16_seed_aggregated_survival": SURVIVAL_SEED_AGG,
    "ig15_sparse16_seed_aggregated_action_behavior": ACTION_BEHAVIOR_SEED_AGG,
    "ig15_sparse16_entropy_action0_timeseries": ENTROPY_ACTION0_SEED_AGG,
    "ig15_sparse16_final_entropy_action0": ENTROPY_ACTION0_FINAL_SEED_AGG,
    "ig15_sparse16_agent_imbalance_timeseries": AGENT_IMBALANCE_SEED_AGG,
    "ig15_sparse16_final_agent_imbalance": AGENT_IMBALANCE_FINAL_SEED_AGG,
    "ig15_sparse16_joint_coordination_coverage": JOINT_COORDINATION_COVERAGE,
    "ig15_sparse16_joint_coordination_timeseries": JOINT_COORDINATION_SEED_AGG,
    "ig15_sparse16_final_joint_coordination": JOINT_COORDINATION_FINAL_SEED_AGG,
    "ig15_sparse16_gate_calibration_coverage": GATE_CALIBRATION_COVERAGE,
    "ig15_sparse16_gate_calibration_timeseries": GATE_CALIBRATION_SEED_AGG,
    "ig15_sparse16_final_gate_calibration": GATE_CALIBRATION_FINAL_SEED_AGG,
}
for name, table in tables_to_export.items():
    path = FIG_DIR / f"{safe_name(name)}.csv"
    table.to_csv(path, index=False)
    print(f"Saved table: {path}")


Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_expected_coverage.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_metric_availability.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_action_profile.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_comparison_specs.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_comparison_deltas.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_largest_changes.csv
Saved table: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/ig15_sparse16_action0_count_summ

## Optional Exact Action-ID Histograms

The scalar history cache cannot reconstruct a full histogram over every discrete action id. It can compare action 0 vs non-idle, gate behavior, joint non-idle counts, illegal action rates, and entropy. If a run logged rollout trace tables with `trace_rollout_actions = true`, this section can fetch those tables and compare exact action-id frequencies for the selected comparison.

In [37]:
def _trace_table_ref_path(value):
    if isinstance(value, dict):
        return value.get("path") or value.get("artifact_path")
    if isinstance(value, str) and value.endswith(".table.json"):
        return value
    return None


def _read_wandb_table_json(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    columns = payload.get("columns") or payload.get("schema", {}).get("columns")
    data = payload.get("data")
    if columns is None or data is None:
        raise ValueError(f"Not a recognized W&B table JSON: {path}")
    return pd.DataFrame(data, columns=columns)


def fetch_trace_tables_from_wandb(selected):
    if not FETCH_TRACE_TABLES_FROM_WANDB:
        print("Trace table fetch disabled. Set FETCH_TRACE_TABLES_FROM_WANDB = True to try W&B table download.")
        return pd.DataFrame()
    try:
        import wandb
    except ImportError as exc:
        raise ImportError("Install wandb first to fetch trace tables.") from exc
    api = wandb.Api(timeout=WANDB_API_TIMEOUT)
    frames = []
    for idx, row in enumerate(selected.to_dict("records"), start=1):
        run_name = row["run_name"]
        run_id = row["run_id"]
        print(f"[{idx:>3}/{len(selected)}] trace tables: {run_name}", flush=True)
        try:
            wb_run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
            for history_row in wb_run.scan_history(keys=["_step", *TRACE_TABLE_KEYS], page_size=2000):
                for key in TRACE_TABLE_KEYS:
                    table_path = _trace_table_ref_path(history_row.get(key))
                    if not table_path:
                        continue
                    target_dir = TRACE_CACHE_DIR / safe_name(run_id)
                    target_dir.mkdir(parents=True, exist_ok=True)
                    downloaded = wb_run.file(table_path).download(root=str(target_dir), replace=False)
                    table = _read_wandb_table_json(downloaded.name)
                    table["trace_key"] = key
                    table["trace_step"] = history_row.get("_step")
                    table["run_name"] = run_name
                    table["run_id"] = run_id
                    frames.append(table)
        except Exception as exc:
            print(f"    skipped: {type(exc).__name__}: {exc}")
    if not frames:
        return pd.DataFrame()
    traces = pd.concat(frames, ignore_index=True, sort=False)
    meta_cols = [
        "run_name",
        "run_id",
        "experiment",
        "comparison_group",
        "family",
        "family_label",
        "seed",
        "design",
        "control_axis",
        "control_value",
        "control_label",
        "entropy_schedule",
    ]
    return traces.merge(selected_runs[meta_cols], on=["run_name", "run_id"], how="left")


def action_trace_to_long(trace_df):
    if trace_df.empty:
        return pd.DataFrame()
    action_cols = [col for col in trace_df.columns if re.match(r"^action_id_agent_\d+$", str(col))]
    if not action_cols:
        print("Trace tables were found, but no action_id_agent_* columns were present.")
        return pd.DataFrame()
    id_cols = [col for col in [
        "source", "rollout", "run_name", "run_id", "experiment", "comparison_group", "family", "family_label",
        "seed", "design", "control_axis", "control_value", "control_label", "entropy_schedule",
        "trace_key", "trace_step", "global_step", "step", "env_idx", "episode", "episode_step", "non_idle_agents", "done",
    ] if col in trace_df.columns]
    long = trace_df[id_cols + action_cols].melt(
        id_vars=id_cols,
        value_vars=action_cols,
        var_name="agent_col",
        value_name="action_id",
    )
    long["agent"] = long["agent_col"].str.extract(r"^action_id_(agent_\d+)$")[0]
    long["action_id"] = pd.to_numeric(long["action_id"], errors="coerce")
    long = long.dropna(subset=["agent", "action_id"]).drop(columns=["agent_col"])

    decoded_cols = [col for col in trace_df.columns if re.match(r"^action_decoded_agent_\d+$", str(col))]
    if decoded_cols:
        decoded_long = trace_df[id_cols + decoded_cols].melt(
            id_vars=id_cols,
            value_vars=decoded_cols,
            var_name="agent_col",
            value_name="action_decoded",
        )
        decoded_long["agent"] = decoded_long["agent_col"].str.extract(r"^action_decoded_(agent_\d+)$")[0]
        decoded_long = decoded_long.drop(columns=["agent_col"]).dropna(subset=["agent"])
        long = long.merge(decoded_long, on=[*id_cols, "agent"], how="left")
    else:
        long["action_decoded"] = ""
    return long


def plot_trace_action_histogram(trace_action_long, top_n=20, comparison_title=SELECTED_COMPARISON):
    if trace_action_long.empty:
        print("No exact action-id traces available to plot.")
        return None
    spec = resolve_selected_comparison(comparison_title)
    keep = trace_action_long["family"].isin([spec["left_family"], spec["right_family"]])
    data = trace_action_long[keep].copy()
    if data.empty:
        print("Trace data exists, but not for the selected comparison.")
        return None
    side_labels = {spec["left_family"]: spec["left_label"], spec["right_family"]: spec["right_label"]}
    data["side_label"] = data["family"].map(side_labels)
    counts = data.groupby(["side_label", "agent", "action_id"], dropna=False, as_index=False).size()
    counts["fraction"] = counts.groupby(["side_label", "agent"], dropna=False)["size"].transform(lambda values: values / max(values.sum(), 1))
    top = counts.sort_values("size", ascending=False).groupby(["side_label", "agent"], dropna=False).head(int(top_n))
    fig = px.bar(
        top,
        x="action_id",
        y="fraction",
        color="side_label",
        barmode="group",
        facet_row="agent",
        hover_data=["size"],
        labels={"action_id": "action id", "fraction": "fraction in trace", "side_label": "run side"},
        title=f"{spec['comparison_title']}: top {top_n} exact action ids from rollout traces",
    )
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=max(650, 260 * data["agent"].nunique()))
    save_figure(fig, f"trace_action_id_histogram_{spec['comparison_id']}")
    if SHOW_FIGURES:
        fig.show()
    return fig


def trace_top_nonzero_action_id_rows(trace_action_long, top_k=TRACE_TOP_K_ACTION_IDS):
    if trace_action_long.empty:
        return pd.DataFrame()
    data = trace_action_long.copy()
    data["action_id"] = pd.to_numeric(data["action_id"], errors="coerce")
    data = data.dropna(subset=["action_id"])
    data = data[data["action_id"].astype(int) != 0].copy()
    if data.empty:
        return pd.DataFrame()
    data["action_id"] = data["action_id"].astype(int)
    decoded = data.get("action_decoded", pd.Series("", index=data.index)).fillna("").astype(str).str.strip()
    data["action_label"] = data["action_id"].astype(str)
    has_decoded = decoded.ne("")
    data.loc[has_decoded, "action_label"] = (
        data.loc[has_decoded, "action_id"].astype(str)
        + " | "
        + decoded[has_decoded].str.slice(0, 90)
    )

    per_run_cols = [
        "run_name", "run_id", "experiment", "comparison_group", "family", "family_label", "seed",
        "design", "control_axis", "control_value", "control_label", "entropy_schedule", "agent", "action_id",
    ]
    per_run = data.groupby(per_run_cols, dropna=False, as_index=False).agg(
        action_count=("action_id", "size"),
        action_label=("action_label", "first"),
        trace_keys=("trace_key", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist()) if "trace_key" in data.columns else []),
    )
    per_run["nonzero_total_for_run_agent"] = per_run.groupby(["run_id", "agent"], dropna=False)["action_count"].transform("sum")
    per_run["fraction_of_nonzero"] = per_run["action_count"] / per_run["nonzero_total_for_run_agent"].replace(0, np.nan)

    condition_cols = [
        "experiment", "comparison_group", "family", "family_label", "design", "control_axis", "control_value",
        "control_label", "entropy_schedule", "agent", "action_id",
    ]
    rows = per_run.groupby(condition_cols, dropna=False, as_index=False).agg(
        mean_fraction_of_nonzero=("fraction_of_nonzero", "mean"),
        std_fraction_of_nonzero=("fraction_of_nonzero", "std"),
        total_action_count=("action_count", "sum"),
        mean_action_count_per_run=("action_count", "mean"),
        n_runs=("run_id", "nunique"),
        n_seeds=("seed", "nunique"),
        seeds=("seed", _seed_list),
        runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        action_label=("action_label", "first"),
    )
    rows["std_fraction_of_nonzero"] = rows["std_fraction_of_nonzero"].fillna(0.0)
    rows["condition_label"] = rows["family_label"]
    rows = rows.sort_values([
        "experiment", "control_value", "design", "condition_label", "agent",
        "mean_fraction_of_nonzero", "total_action_count",
    ], ascending=[True, True, True, True, True, False, False])
    rows["rank_within_condition_agent"] = rows.groupby(
        ["experiment", "family", "agent"], dropna=False
    ).cumcount() + 1
    return rows[rows["rank_within_condition_agent"] <= int(top_k)].reset_index(drop=True)


def plot_top_nonzero_action_ids_by_condition(top_action_rows, experiment, title, save_name, top_k=TRACE_TOP_K_ACTION_IDS):
    if top_action_rows is None or top_action_rows.empty or "experiment" not in top_action_rows.columns:
        print(f"No non-zero exact action-id trace data for {experiment}.")
        return None
    data = top_action_rows[top_action_rows["experiment"] == experiment].copy()
    if data.empty:
        print(f"No non-zero exact action-id trace data for {experiment}.")
        return None
    data["action_id_label"] = data["action_id"].astype(int).astype(str)
    condition_order = (
        data[["condition_label", "control_value", "design"]]
        .drop_duplicates()
        .sort_values(["control_value", "design", "condition_label"])["condition_label"]
        .tolist()
    )
    agent_order = sorted(data["agent"].dropna().astype(str).unique().tolist())
    fig = px.bar(
        data,
        x="action_id_label",
        y="mean_fraction_of_nonzero",
        color="condition_label",
        error_y="std_fraction_of_nonzero",
        facet_row="agent",
        facet_col="condition_label",
        category_orders={"condition_label": condition_order, "agent": agent_order},
        hover_data={
            "action_id": True,
            "action_label": True,
            "mean_fraction_of_nonzero": ":.4f",
            "std_fraction_of_nonzero": ":.4f",
            "total_action_count": True,
            "mean_action_count_per_run": ":.2f",
            "n_runs": True,
            "n_seeds": True,
            "seeds": True,
            "rank_within_condition_agent": True,
        },
        labels={
            "action_id_label": "non-zero action id",
            "mean_fraction_of_nonzero": "fraction among non-zero trace actions",
            "condition_label": "condition",
        },
        title=title,
    )
    fig.update_xaxes(matches=None, tickangle=45, title_text="action id")
    fig.update_yaxes(matches=None, rangemode="tozero")
    fig.update_layout(
        template="plotly_white",
        height=max(720, 270 * max(len(agent_order), 1)),
        width=max(1500, 260 * max(len(condition_order), 1)),
        showlegend=False,
        margin={"l": 80, "r": 40, "t": 105, "b": 100},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return fig


trace_tables = fetch_trace_tables_from_wandb(selected_runs)
trace_action_long = action_trace_to_long(trace_tables)
fig_trace_actions = plot_trace_action_histogram(trace_action_long, comparison_title=selected_spec["comparison_title"])

TRACE_TOP_NONZERO_ACTION_IDS = trace_top_nonzero_action_id_rows(trace_action_long, top_k=TRACE_TOP_K_ACTION_IDS)
print(f"Top {TRACE_TOP_K_ACTION_IDS} non-zero exact action IDs by agent and condition:")
if TRACE_TOP_NONZERO_ACTION_IDS.empty:
    print("No top action-ID table available. This requires runs logged with trace_rollout_actions=true and fetched trace tables.")
else:
    display(TRACE_TOP_NONZERO_ACTION_IDS.groupby(["experiment", "family_label", "agent"], dropna=False).agg(
        top_actions=("action_id", lambda values: list(values)),
        n_seeds=("n_seeds", "max"),
        seeds=("seeds", "first"),
    ).reset_index())
    trace_top_path = FIG_DIR / "ig15_sparse16_top_nonzero_action_ids.csv"
    TRACE_TOP_NONZERO_ACTION_IDS.to_csv(trace_top_path, index=False)
    print(f"Saved table: {trace_top_path}")

fig_top_nonzero_action_ids_ig15 = plot_top_nonzero_action_ids_by_condition(
    TRACE_TOP_NONZERO_ACTION_IDS,
    "intervention_gate_15",
    f"Intervention gate 15: top {TRACE_TOP_K_ACTION_IDS} non-zero action IDs by agent and condition",
    "top_nonzero_action_ids_intervention_gate_15",
    top_k=TRACE_TOP_K_ACTION_IDS,
)
fig_top_nonzero_action_ids_sparse16 = plot_top_nonzero_action_ids_by_condition(
    TRACE_TOP_NONZERO_ACTION_IDS,
    "phase4_sparse_control_16",
    f"Phase4 sparse control 16: top {TRACE_TOP_K_ACTION_IDS} non-zero action IDs by agent and condition",
    "top_nonzero_action_ids_phase4_sparse_control_16",
    top_k=TRACE_TOP_K_ACTION_IDS,
)


Trace table fetch disabled. Set FETCH_TRACE_TABLES_FROM_WANDB = True to try W&B table download.
No exact action-id traces available to plot.
Top 10 non-zero exact action IDs by agent and condition:
No top action-ID table available. This requires runs logged with trace_rollout_actions=true and fetched trace tables.
No non-zero exact action-id trace data for intervention_gate_15.
No non-zero exact action-id trace data for phase4_sparse_control_16.
